# PoliMillionaire Final Notebook

Group members: **name - email**, **name - email**, **name - email**

Video link: **add link here**

Coding assistant statement: **fill this honestly before submission**


## Setup


In [1]:
import gc
import importlib.util
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "polimillionaire" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the project folder.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
for path in [REPO_ROOT / "src", REPO_ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

os.environ.setdefault("HF_HOME", str(REPO_ROOT / "data" / "hf_home"))
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(REPO_ROOT / "data" / "hf_home" / "hub"))
print("Repo:", REPO_ROOT)
print("HF cache:", os.environ["HF_HOME"])


Repo: C:\Users\sjuan\Desktop\NLP\project\nlp-polimillionaire
HF cache: C:\Users\sjuan\Desktop\NLP\project\nlp-polimillionaire\data\hf_home


## Dependencies


In [2]:
import importlib.util
import subprocess
import sys

INSTALL_DEPENDENCIES = False
packages = [
    "requests>=2.31",
    "python-dotenv>=1.0",
    "torch>=2.2",
    "accelerate>=0.30",
    "transformers>=5.7.0",
    "huggingface_hub>=0.24",
    "bitsandbytes>=0.43",
    "ddgs>=9.0",
    "httpx>=0.27",
    "trafilatura>=1.12",
    "langchain-community>=0.3.0",
    "langchain-core>=0.3.0",
    "langchain-text-splitters>=0.3.0",
    "sentence-transformers>=3.0",
    "rank_bm25>=0.2.2",
]

if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *packages])

for module in ["requests", "torch", "transformers", "accelerate", "bitsandbytes", "ddgs", "trafilatura", "sentence_transformers", "rank_bm25"]:
    print(f"{module:22s}", bool(importlib.util.find_spec(module)))


requests               True
torch                  True
transformers           True
accelerate             True
bitsandbytes           True
ddgs                   True
trafilatura            True
sentence_transformers  True
rank_bm25              True


## Included API Client


In [3]:
import requests
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from urllib.parse import urljoin


class MillionaireError(Exception):
    pass


class AuthenticationError(MillionaireError):
    pass


class GameStatus(Enum):
    IN_PROGRESS = "in_progress"
    COMPLETED = "completed"
    FAILED = "failed"
    TIMEOUT = "timeout"


@dataclass
class User:
    id: int
    username: str
    role: str

    @classmethod
    def from_dict(cls, data):
        return cls(id=data["id"], username=data["username"], role=data["role"])


@dataclass
class ApiOption:
    id: int
    text: str | None = None

    @classmethod
    def from_dict(cls, data):
        return cls(id=int(data["id"]), text=data.get("text"))


@dataclass
class ApiQuestion:
    id: int
    text: str | None = None
    options: list[ApiOption] = field(default_factory=list)
    level: int = 0

    @classmethod
    def from_dict(cls, data):
        return cls(
            id=int(data["id"]),
            text=data.get("text"),
            options=[ApiOption.from_dict(item) for item in data.get("options", [])],
            level=int(data.get("level", 0) or 0),
        )


@dataclass
class Competition:
    id: int
    name: str
    description: str | None = None
    max_levels: int = 15
    is_infinite: bool = False
    question_count: int | None = None

    @classmethod
    def from_dict(cls, data):
        return cls(
            id=int(data["id"]),
            name=str(data["name"]),
            description=data.get("description"),
            max_levels=int(data.get("maxLevels", 15) or 15),
            is_infinite=bool(data.get("isInfinite", False)),
            question_count=data.get("questionCount"),
        )


@dataclass
class ApiGameState:
    session_id: int
    competition: Competition
    status: GameStatus
    earned_amount: float
    current_level: int
    question_deadline: datetime | None = None
    question: ApiQuestion | None = None
    money_pyramid: list = field(default_factory=list)
    mode: str = "text"

    @classmethod
    def from_dict(cls, data):
        deadline = data.get("questionDeadline")
        if deadline:
            try:
                deadline = datetime.fromisoformat(deadline.replace("Z", "+00:00"))
            except Exception:
                deadline = None
        return cls(
            session_id=int(data.get("sessionId", data.get("id", 0))),
            competition=Competition.from_dict(data["competition"]),
            status=GameStatus(data.get("status", "in_progress")),
            earned_amount=float(data.get("earnedAmount", 0) or 0),
            current_level=int(data.get("currentLevel", 1) or 1),
            question_deadline=deadline,
            question=ApiQuestion.from_dict(data["question"]) if data.get("question") else None,
            money_pyramid=data.get("moneyPyramid", []),
            mode=data.get("mode", "text"),
        )

    @property
    def in_progress(self):
        return self.status == GameStatus.IN_PROGRESS


@dataclass
class ApiAnswerResult:
    correct: bool | None = None
    game_over: bool = False
    earned_amount: float = 0.0
    timed_out: bool = False
    status: str | None = None
    current_level: int | None = None
    question: ApiQuestion | None = None
    question_deadline: datetime | None = None

    @classmethod
    def from_dict(cls, data):
        deadline = data.get("questionDeadline")
        if deadline:
            try:
                deadline = datetime.fromisoformat(deadline.replace("Z", "+00:00"))
            except Exception:
                deadline = None
        return cls(
            correct=data.get("correct"),
            game_over=bool(data.get("gameOver", False)),
            earned_amount=float(data.get("earnedAmount", 0) or 0),
            timed_out=bool(data.get("timedOut", False)),
            status=data.get("status"),
            current_level=data.get("currentLevel"),
            question=ApiQuestion.from_dict(data["question"]) if data.get("question") else None,
            question_deadline=deadline,
        )


class BaseClient:
    def __init__(self, base_url, timeout=30):
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self.session = requests.Session()

    @property
    def is_authenticated(self):
        return "polimillionaire_auth" in self.session.cookies

    def request(self, method, endpoint, data=None, params=None, auth_required=True, raw=False):
        if auth_required and not self.is_authenticated:
            raise AuthenticationError("Authentication required")
        url = urljoin(self.base_url + "/", endpoint.lstrip("/"))
        try:
            response = self.session.request(method, url, json=data, params=params, timeout=self.timeout)
        except requests.RequestException as exc:
            raise MillionaireError(f"API request failed: {exc}") from exc
        if raw and response.status_code in (200, 201, 204):
            return response.content
        payload = response.json() if response.text else {}
        if response.status_code in (200, 201, 204):
            return payload
        message = payload.get("message", payload.get("error", f"HTTP {response.status_code}"))
        if response.status_code == 401:
            raise AuthenticationError(message)
        raise MillionaireError(message)

    def get(self, endpoint, **kwargs):
        return self.request("GET", endpoint, **kwargs)

    def post(self, endpoint, **kwargs):
        return self.request("POST", endpoint, **kwargs)


class GameSession:
    def __init__(self, client, state):
        self._client = client
        self._state = state

    @property
    def session_id(self):
        return self._state.session_id

    @property
    def current_question(self):
        return self._state.question

    @property
    def current_level(self):
        return self._state.current_level

    @property
    def earned_amount(self):
        return self._state.earned_amount

    @property
    def in_progress(self):
        return self._state.in_progress

    def answer(self, option_id):
        data = self._client.post(f"/api/game/{self.session_id}/answer", data={"optionId": int(option_id)})
        result = ApiAnswerResult.from_dict(data)
        if result.question:
            self._state.question = result.question
            self._state.current_level = result.current_level or self._state.current_level
            self._state.earned_amount = result.earned_amount
            self._state.question_deadline = result.question_deadline
        else:
            self._state.earned_amount = result.earned_amount
            self._state.question = None
            if result.game_over:
                self._state.status = GameStatus.COMPLETED if result.correct else GameStatus.FAILED
        return result


class GameModule:
    def __init__(self, client):
        self._client = client

    def start(self, competition_id, mode="text"):
        data = self._client.post("/api/game/start", data={"competitionId": int(competition_id), "mode": mode})
        return GameSession(self._client, ApiGameState.from_dict(data))


class CompetitionsModule:
    def __init__(self, client):
        self._client = client

    def list_all(self):
        data = self._client.get("/api/competitions")
        return [Competition.from_dict(item) for item in data.get("competitions", [])]


class MillionaireClient:
    def __init__(self, base_url, timeout=30):
        self._base = BaseClient(base_url, timeout=timeout)
        self.game = GameModule(self._base)
        self.competitions = CompetitionsModule(self._base)

    @property
    def is_authenticated(self):
        return self._base.is_authenticated

    def login(self, username, password):
        data = self._base.post("/api/auth/login", data={"username": username, "password": password}, auth_required=False)
        return User.from_dict(data["user"])


## Included Model Harness


In [4]:
from __future__ import annotations



"""Small dataclasses shared by the assignment notebook helpers."""

from dataclasses import dataclass, field
from typing import Any


@dataclass(frozen=True)
class AnswerOption:
    id: int
    text: str


@dataclass(frozen=True)
class Question:
    id: int
    text: str
    options: list[AnswerOption]
    level: int = 0
    metadata: dict[str, Any] = field(default_factory=dict)

    def valid_option_ids(self) -> set[int]:
        return {option.id for option in self.options}

    def first_option(self) -> AnswerOption:
        if not self.options:
            raise ValueError("Question has no answer options")
        return self.options[0]

    def get_option(self, option_id: int) -> AnswerOption | None:
        for option in self.options:
            if option.id == option_id:
                return option
        return None

    def require_option(self, option_id: int) -> AnswerOption:
        return self.get_option(option_id) or self.first_option()


@dataclass
class AnswerPrediction:
    option_id: int
    answer_text: str
    confidence: float | None = None
    reasoning: str | None = None
    metadata: dict[str, Any] = field(default_factory=dict)

"""Strategies, local model loading, prompting, and output parsing."""

import json
import math
import random
import re
from abc import ABC, abstractmethod
from dataclasses import dataclass, fields
from typing import Any, Callable, Protocol
import asyncio
import hashlib
import time
import logging

from packaging.version import Version



@dataclass(frozen=True)
class RouteDecision:
    route: str
    reason: str


class BaseStrategy(ABC):
    name = "base"

    @abstractmethod
    def answer(self, question: Question) -> AnswerPrediction:
        raise NotImplementedError


class RandomStrategy(BaseStrategy):
    name = "random"

    def __init__(self, seed: int | None = None):
        self._rng = random.Random(seed)

    def answer(self, question: Question) -> AnswerPrediction:
        option = self._rng.choice(question.options)
        return AnswerPrediction(
            option_id=option.id,
            answer_text=option.text,
            confidence=1.0 / max(1, len(question.options)),
            reasoning="Random option selected.",
            metadata={"strategy": self.name},
        )


class HeuristicStrategy(BaseStrategy):
    name = "heuristic"

    def answer(self, question: Question) -> AnswerPrediction:
        question_terms = set(_words(question.text))
        option = max(
            question.options,
            key=lambda item: len(question_terms & set(_words(item.text))),
        )
        return AnswerPrediction(
            option_id=option.id,
            answer_text=option.text,
            confidence=0.35,
            reasoning="Simple word-overlap heuristic selected the option.",
            metadata={"strategy": self.name},
        )


class CalculatorStrategy(BaseStrategy):
    name = "calculator"

    def __init__(self, fallback_strategy: BaseStrategy | None = None):
        self.fallback_strategy = fallback_strategy

    def answer(self, question: Question) -> AnswerPrediction:
        solved = _solve_math_question(question)
        if solved is not None:
            option, value, method = solved
            return AnswerPrediction(
                option_id=option.id,
                answer_text=option.text,
                confidence=1.0,
                reasoning=f"Calculated value: {value}.",
                metadata={
                    "strategy": self.name,
                    "fallback": False,
                    "tool": "calculator",
                    "calculated_value": value,
                    "calculation_method": method,
                },
            )
        if self.fallback_strategy is not None:
            prediction = self.fallback_strategy.answer(question)
            prediction.metadata["calculator_attempted"] = True
            return prediction
        option = question.first_option()
        return AnswerPrediction(
            option_id=option.id,
            answer_text=option.text,
            confidence=0.0,
            reasoning="Could not calculate a matching option.",
            metadata={"strategy": self.name, "fallback": True},
        )


def route_question(question: Question) -> RouteDecision:
    question_lowered = question.text.lower()
    text = f"{question.text} " + " ".join(option.text for option in question.options)
    lowered = text.lower()
    if _looks_recent_or_report(lowered):
        return RouteDecision("rag", "recent_or_report")
    if _looks_like_math(question_lowered):
        return RouteDecision("direct", "math")
    if _has_negation_trap(question_lowered):
        return RouteDecision("direct", "negation")
    if _looks_factual(lowered):
        return RouteDecision("rag", "factual")
    return RouteDecision("fallback", "general")


def _retrieval_query(question: Question) -> str:
    option_text = " ".join(option.text for option in question.options)
    full_query = " ".join(f"{question.text} {option_text}".split())
    lowered = question.text.lower()
    option_helpful = (
        "which of the following" in lowered
        or "which of these" in lowered
        or "which term" in lowered
        or "first" in lowered
        or "best describes" in lowered
        or "according" in lowered
    )
    if _looks_recent_or_report(full_query.lower()) or option_helpful:
        return full_query[:500]
    return question.text


class RoutedStrategy(BaseStrategy):
    name = "routed"

    def __init__(
        self,
        direct_strategy: BaseStrategy,
        rag_strategy: BaseStrategy | None = None,
        fallback_strategy: BaseStrategy | None = None,
        low_confidence_strategy: BaseStrategy | None = None,
        rag_min_confidence: float | None = None,
    ):
        self.direct_strategy = direct_strategy
        self.rag_strategy = rag_strategy
        self.fallback_strategy = fallback_strategy or direct_strategy
        self.low_confidence_strategy = low_confidence_strategy
        self.rag_min_confidence = rag_min_confidence

    def answer(self, question: Question) -> AnswerPrediction:
        decision = route_question(question)
        if decision.route == "rag" and self.rag_strategy is not None:
            strategy = self.rag_strategy
        elif decision.route == "direct":
            strategy = self.direct_strategy
        else:
            strategy = self.fallback_strategy
        prediction = strategy.answer(question)
        prediction.metadata["route"] = decision.route
        prediction.metadata["route_reason"] = decision.reason
        prediction.metadata["routed_to"] = getattr(strategy, "name", strategy.__class__.__name__)
        if (
            decision.route == "rag"
            and self.low_confidence_strategy is not None
            and self.rag_min_confidence is not None
            and (prediction.confidence is None or prediction.confidence < self.rag_min_confidence)
        ):
            backup = self.low_confidence_strategy.answer(question)
            backup.metadata["route"] = decision.route
            backup.metadata["route_reason"] = decision.reason
            backup.metadata["routed_to"] = getattr(self.low_confidence_strategy, "name", "backup")
            backup.metadata["backup_for_low_confidence_rag"] = True
            backup.metadata["rag_prediction"] = {
                "option_id": prediction.option_id,
                "confidence": prediction.confidence,
                "reasoning": prediction.reasoning,
                "metadata": prediction.metadata,
            }
            return backup
        return prediction


class LocalLLM(Protocol):
    model_name: str

    def generate(self, prompt: str, **kwargs: object) -> str:
        ...


class FakeLLM:
    def __init__(self, responses: list[str] | None = None, model_name: str = "fake-llm"):
        self.responses = list(responses or ['{"option_id": 0, "confidence": 0.5, "reason": "fake"}'])
        self.model_name = model_name
        self.prompts: list[str] = []
        self.calls: list[dict[str, object]] = []

    def generate(self, prompt: str, **kwargs: object) -> str:
        self.prompts.append(prompt)
        self.calls.append(dict(kwargs))
        if len(self.responses) == 1:
            return self.responses[0]
        return self.responses.pop(0)


class LocalLLMVariant:
    def __init__(self, llm: LocalLLM, variant_name: str, **generation_overrides: Any):
        self.llm = llm
        self.model_name = variant_name
        self.generation_overrides = {
            key: value for key, value in generation_overrides.items() if value is not None
        }

    @property
    def device_summary(self) -> str:
        return getattr(self.llm, "device_summary", "unknown")

    def generate(self, prompt: str, **kwargs: object) -> str:
        merged = dict(kwargs)
        merged.update(self.generation_overrides)
        return self.llm.generate(prompt, **merged)


@dataclass
class GemmaLLMConfig:
    model_id: str = "google/gemma-4-E2B-it"
    inference_backend: str = "auto_model"
    device_map: str = "auto"
    dtype: str = "auto"
    max_new_tokens: int = 8
    temperature: float = 0.0
    do_sample: bool = False
    num_beams: int = 1
    seed: int | None = 42
    generation_max_time_seconds: float | None = 18.0
    timeout_seconds: float | None = 25.0
    quantize_4bit: bool = False


@dataclass
class QwenLLMConfig:
    model_id: str = "Qwen/Qwen3.5-2B"
    device_map: str = "auto"
    dtype: str = "auto"
    max_new_tokens: int = 256
    temperature: float = 1.0
    do_sample: bool = True
    top_p: float = 0.95
    top_k: int = 20
    seed: int | None = 42
    enable_thinking: bool = True
    generation_max_time_seconds: float | None = None
    timeout_seconds: float | None = 25.0
    quantize_4bit: bool = False


class GemmaLLM:
    def __init__(self, config: GemmaLLMConfig | None = None, **overrides: Any):
        self.config = config or GemmaLLMConfig()
        for key, value in overrides.items():
            if hasattr(self.config, key):
                setattr(self.config, key, value)
        self.model_name = self.config.model_id
        self._model: Any = None
        self._processor: Any = None
        self._tokenizer: Any = None
        self._pipeline: Any = None

    @property
    def is_loaded(self) -> bool:
        return self._model is not None or self._pipeline is not None

    @property
    def device_summary(self) -> str:
        if self._model is not None:
            device_map = getattr(self._model, "hf_device_map", None)
            if device_map:
                return str(device_map)
            try:
                return str(next(self._model.parameters()).device)
            except Exception:
                return "unknown"
        if self._pipeline is not None:
            return str(getattr(self._pipeline, "device", "pipeline"))
        return "not_loaded"

    def generate(self, prompt: str, **kwargs: Any) -> str:
        self._load()
        seed = kwargs.pop("seed", self.config.seed)
        self._seed(seed)
        generation_kwargs = self._generation_kwargs(kwargs)
        if self.config.inference_backend == "auto_model":
            text = self._generate_auto_model(prompt, generation_kwargs)
        elif self.config.inference_backend == "pipeline_any_to_any":
            text = self._generate_pipeline(prompt, generation_kwargs)
        else:
            raise ValueError(f"Unsupported inference_backend: {self.config.inference_backend}")
        return text.strip()

    def unload(self) -> None:
        self._model = None
        self._processor = None
        self._tokenizer = None
        self._pipeline = None
        _clear_torch_memory()
    
    def invoke(self, input_data: Any, config: Any = None) -> Any:
        from langchain_core.outputs import ChatResult, ChatGeneration
        from langchain_core.messages import BaseMessage
        
        if hasattr(input_data, "to_string"):
            prompt_text = input_data.to_string()
        elif isinstance(input_data, list) and len(input_data) > 0:
            prompt_text = getattr(input_data[-1], "content", str(input_data))
        else:
            prompt_text = str(input_data)
            

        generated_text = self.generate(prompt_text)

        return str(generated_text)
        

    def _load(self) -> None:
        if self.is_loaded:
            return
        if self.config.inference_backend == "auto_model":
            self._load_auto_model()
        elif self.config.inference_backend == "pipeline_any_to_any":
            self._load_pipeline()
        else:
            raise ValueError(f"Unsupported inference_backend: {self.config.inference_backend}")

    def _load_auto_model(self) -> None:
        from transformers import AutoModelForCausalLM, AutoProcessor, AutoTokenizer
        import transformers

        _require_supported_transformers(transformers.__version__)
        try:
            self._processor = AutoProcessor.from_pretrained(self.config.model_id)
        except Exception:
            self._tokenizer = AutoTokenizer.from_pretrained(self.config.model_id, extra_special_tokens={})
        self._model = AutoModelForCausalLM.from_pretrained(
            self.config.model_id,
            device_map=self.config.device_map,
            dtype=self.config.dtype,
            **_quantization_kwargs(self.config.quantize_4bit),
        )
        self._model.eval()

    def _load_pipeline(self) -> None:
        from transformers import pipeline

        kwargs: dict[str, Any] = {
            "task": "any-to-any",
            "model": self.config.model_id,
            "device_map": self.config.device_map,
            "dtype": self.config.dtype,
        }
        if self.config.quantize_4bit:
            kwargs["model_kwargs"] = _quantization_kwargs(True)
        self._pipeline = pipeline(**kwargs)

    def _generation_kwargs(self, overrides: dict[str, Any]) -> dict[str, Any]:
        kwargs = {
            "max_new_tokens": self.config.max_new_tokens,
            "temperature": self.config.temperature,
            "do_sample": self.config.do_sample,
            "num_beams": self.config.num_beams,
        }
        if self.config.generation_max_time_seconds is not None:
            kwargs["max_time"] = self.config.generation_max_time_seconds
        kwargs.update({key: value for key, value in overrides.items() if value is not None})
        if not kwargs["do_sample"]:
            kwargs.pop("temperature", None)
        return kwargs

    def _seed(self, seed: int | None) -> None:
        if seed is None:
            return
        try:
            import torch

            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)
        except Exception:
            pass

    def _generate_auto_model(self, prompt: str, generation_kwargs: dict[str, Any]) -> str:
        import torch

        processor = self._processor or self._tokenizer
        inputs = _tokenize_prompt(processor, prompt)
        try:
            device = next(self._model.parameters()).device
            inputs = {key: value.to(device) if hasattr(value, "to") else value for key, value in inputs.items()}
        except Exception:
            pass
        input_length = inputs["input_ids"].shape[-1]
        with torch.inference_mode():
            output_ids = self._model.generate(**inputs, **generation_kwargs)
        generated_ids = output_ids[0][input_length:]
        return processor.decode(generated_ids, skip_special_tokens=True)

    def _generate_pipeline(self, prompt: str, generation_kwargs: dict[str, Any]) -> str:
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        result = self._pipeline(messages, return_full_text=False, generate_kwargs=generation_kwargs)
        if isinstance(result, list) and result:
            item = result[0]
            if isinstance(item, dict):
                return str(item.get("generated_text", ""))
            return str(item)
        return str(result)


class QwenLLM:
    def __init__(self, config: QwenLLMConfig | None = None, **overrides: Any):
        self.config = config or QwenLLMConfig()
        for key, value in overrides.items():
            if hasattr(self.config, key):
                setattr(self.config, key, value)
        self.model_name = self.config.model_id
        self._model: Any = None
        self._processor: Any = None

    @property
    def is_loaded(self) -> bool:
        return self._model is not None

    @property
    def device_summary(self) -> str:
        if self._model is None:
            return "not_loaded"
        device_map = getattr(self._model, "hf_device_map", None)
        if device_map:
            return str(device_map)
        try:
            return str(next(self._model.parameters()).device)
        except Exception:
            return "unknown"

    def generate(self, prompt: str, **kwargs: Any) -> str:
        self._load()
        seed = kwargs.pop("seed", self.config.seed)
        self._seed(seed)
        generation_kwargs = self._generation_kwargs(kwargs)
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        inputs = self._processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=self.config.enable_thinking,
        )
        try:
            device = next(self._model.parameters()).device
            inputs = {key: value.to(device) if hasattr(value, "to") else value for key, value in inputs.items()}
        except Exception:
            pass

        import torch

        input_length = inputs["input_ids"].shape[-1]
        with torch.inference_mode():
            output_ids = self._model.generate(**inputs, **generation_kwargs)
        generated_ids = output_ids[0][input_length:]
        return self._processor.decode(generated_ids, skip_special_tokens=True).strip()

    def unload(self) -> None:
        self._model = None
        self._processor = None
        _clear_torch_memory()
    
    def invoke(self, input_data: Any, config: Any = None) -> Any:
        from langchain_core.outputs import ChatResult, ChatGeneration
        from langchain_core.messages import BaseMessage
        
        if hasattr(input_data, "to_string"):
            prompt_text = input_data.to_string()
        elif isinstance(input_data, list) and len(input_data) > 0:
            prompt_text = getattr(input_data[-1], "content", str(input_data))
        else:
            prompt_text = str(input_data)
            

        generated_text = self.generate(prompt_text)

        return str(generated_text)

    def _load(self) -> None:
        if self.is_loaded:
            return
        from transformers import AutoModelForImageTextToText, AutoProcessor
        import transformers

        _require_supported_transformers(transformers.__version__)
        self._processor = AutoProcessor.from_pretrained(self.config.model_id)
        self._model = AutoModelForImageTextToText.from_pretrained(
            self.config.model_id,
            device_map=self.config.device_map,
            dtype=self.config.dtype,
            **_quantization_kwargs(self.config.quantize_4bit),
        )
        self._model.eval()

    def _generation_kwargs(self, overrides: dict[str, Any]) -> dict[str, Any]:
        kwargs = {
            "max_new_tokens": self.config.max_new_tokens,
            "do_sample": self.config.do_sample,
        }
        if self.config.do_sample:
            kwargs.update(
                {
                    "temperature": self.config.temperature,
                    "top_p": self.config.top_p,
                    "top_k": self.config.top_k,
                }
            )
        if self.config.generation_max_time_seconds is not None:
            kwargs["max_time"] = self.config.generation_max_time_seconds
        kwargs.update({key: value for key, value in overrides.items() if value is not None})
        if not kwargs.get("do_sample", False):
            kwargs.pop("temperature", None)
            kwargs.pop("top_p", None)
            kwargs.pop("top_k", None)
        return kwargs

    def _seed(self, seed: int | None) -> None:
        if seed is None:
            return
        try:
            import torch

            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)
        except Exception:
            pass


class GemmaStrategy(BaseStrategy):
    name = "gemma"

    def __init__(self, llm: LocalLLM | None = None, model_config: GemmaLLMConfig | dict[str, Any] | None = None):
        if llm is None:
            if isinstance(model_config, dict):
                allowed = {field.name for field in fields(GemmaLLMConfig)}
                model_config = GemmaLLMConfig(**{key: value for key, value in model_config.items() if key in allowed})
            llm = GemmaLLM(model_config)
        self.llm = llm

    def answer(self, question: Question) -> AnswerPrediction:
        raw_text = self.llm.generate(build_prompt(question))
        prediction = parse_answer_prediction(raw_text, question, strategy_name=self.name)
        prediction.metadata["strategy"] = self.name
        prediction.metadata["model_name"] = getattr(self.llm, "model_name", "unknown")
        prediction.metadata["device"] = getattr(self.llm, "device_summary", "unknown")
        return prediction


class QwenStrategy(BaseStrategy):
    name = "qwen3.5_thinking"

    def __init__(self, llm: LocalLLM | None = None, model_config: QwenLLMConfig | dict[str, Any] | None = None):
        if isinstance(model_config, dict):
            allowed = {field.name for field in fields(QwenLLMConfig)}
            model_config = QwenLLMConfig(**{key: value for key, value in model_config.items() if key in allowed})
        self.config = model_config or QwenLLMConfig()
        self.llm = llm or QwenLLM(self.config)

    def answer(self, question: Question) -> AnswerPrediction:
        raw_text = self.llm.generate(build_qwen_prompt(question))
        prediction = parse_answer_prediction(raw_text, question, strategy_name=self.name)
        prediction.metadata["strategy"] = self.name
        prediction.metadata["model_name"] = getattr(self.llm, "model_name", "unknown")
        prediction.metadata["device"] = getattr(self.llm, "device_summary", "unknown")
        prediction.metadata["thinking"] = self.config.enable_thinking
        return prediction


class CouncilStrategy(BaseStrategy):
    name = "council"

    def __init__(
        self,
        llm: LocalLLM | None = None,
        judge_llm: LocalLLM | None = None,
        candidate_llms: list[LocalLLM] | None = None,
        num_votes: int = 3,
        base_seed: int = 100,
        candidate_temperature: float = 0.8,
        candidate_top_p: float = 0.9,
        candidate_max_new_tokens: int = 48,
        judge_max_new_tokens: int = 8,
        max_time_per_call: float | None = None,
        shuffle_options: bool = False,
        judge_scope: str = "any_option",
        rejected_judge_fallback: str = "confidence_weighted",
    ):
        if candidate_llms:
            self.candidate_llms = list(candidate_llms)
            self.llm = llm or self.candidate_llms[0]
        elif llm is not None:
            if num_votes < 1:
                raise ValueError("num_votes must be at least 1")
            self.llm = llm
            self.candidate_llms = [llm] * num_votes
        else:
            raise ValueError("Provide llm or candidate_llms")
        if not self.candidate_llms:
            raise ValueError("num_votes must be at least 1")
        self.judge_llm = judge_llm or self.llm
        self.num_votes = len(self.candidate_llms)
        self.base_seed = base_seed
        self.candidate_temperature = candidate_temperature
        self.candidate_top_p = candidate_top_p
        self.candidate_max_new_tokens = candidate_max_new_tokens
        self.judge_max_new_tokens = judge_max_new_tokens
        self.max_time_per_call = max_time_per_call
        self.shuffle_options = shuffle_options
        if judge_scope not in {"candidate_only", "any_option"}:
            raise ValueError("judge_scope must be 'candidate_only' or 'any_option'")
        self.judge_scope = judge_scope
        if rejected_judge_fallback not in {"confidence_weighted", "primary_candidate"}:
            raise ValueError("rejected_judge_fallback must be 'confidence_weighted' or 'primary_candidate'")
        self.rejected_judge_fallback = rejected_judge_fallback

    def answer(self, question: Question) -> AnswerPrediction:
        votes: list[AnswerPrediction] = []
        for vote_index, candidate_llm in enumerate(self.candidate_llms):
            option_order = self._option_order(question, vote_index)
            raw_text = candidate_llm.generate(
                build_council_vote_prompt(question, option_order),
                max_new_tokens=self.candidate_max_new_tokens,
                do_sample=True,
                temperature=self.candidate_temperature,
                top_p=self.candidate_top_p,
                seed=self.base_seed + vote_index,
                **self._time_kwargs(),
            )
            vote = parse_answer_prediction(raw_text, question, strategy_name="council_vote")
            if not vote.metadata["fallback"]:
                vote.metadata["model_name"] = getattr(candidate_llm, "model_name", "unknown")
                vote.metadata["sample_seed"] = self.base_seed + vote_index
                vote.metadata["option_order"] = [option.id for option in option_order]
                votes.append(vote)

        if not votes:
            return _council_fallback(question, "No valid candidate votes.", [])

        supported_options = {vote.option_id for vote in votes}
        majority_option = _majority_option(votes)
        if majority_option is not None:
            result = _selected_vote_prediction(question, votes, majority_option)
            method = "unanimous_vote" if len(supported_options) == 1 else "majority_vote"
            result.metadata.update(self._metadata(votes, None, method))
            return result

        judge_raw_text = self.judge_llm.generate(
            build_judge_prompt(question, votes, judge_scope=self.judge_scope),
            max_new_tokens=self.judge_max_new_tokens,
            do_sample=False,
            seed=self.base_seed + self.num_votes,
            **self._time_kwargs(),
        )
        judged = parse_answer_prediction(judge_raw_text, question, strategy_name=self.name)
        if not judged.metadata["fallback"] and (
            self.judge_scope == "any_option" or judged.option_id in supported_options
        ):
            judged.metadata.update(self._metadata(votes, judge_raw_text, "judge"))
            judged.metadata["judge_novel_choice"] = judged.option_id not in supported_options
            return judged

        if self.rejected_judge_fallback == "primary_candidate":
            result = _selected_vote_prediction(question, votes, votes[0].option_id)
            result.reasoning = "Primary candidate selected after judge returned an unsupported answer."
            method = "primary_candidate"
        else:
            result = _weighted_vote(question, votes)
            method = "weighted_vote"
        result.metadata.update(self._metadata(votes, judge_raw_text, method))
        result.metadata["judge_rejected"] = True
        result.metadata["judge_option_id"] = None if judged.metadata["fallback"] else judged.option_id
        return result

    def _time_kwargs(self) -> dict[str, float]:
        if self.max_time_per_call is None:
            return {}
        return {"max_time": self.max_time_per_call}

    def _option_order(self, question: Question, vote_index: int) -> list[Any]:
        options = list(question.options)
        if self.shuffle_options:
            random.Random(self.base_seed + vote_index).shuffle(options)
        return options

    def _metadata(self, votes: list[AnswerPrediction], judge_raw_text: str | None, method: str) -> dict[str, Any]:
        return {
            "strategy": self.name,
            "model_name": getattr(self.llm, "model_name", "unknown"),
            "judge_model_name": getattr(self.judge_llm, "model_name", "unknown"),
            "device": getattr(self.llm, "device_summary", "unknown"),
            "decision_method": method,
            "judge_scope": self.judge_scope,
            "rejected_judge_fallback": self.rejected_judge_fallback,
            "candidate_devices": [
                getattr(candidate_llm, "device_summary", "unknown") for candidate_llm in self.candidate_llms
            ],
            "judge_device": getattr(self.judge_llm, "device_summary", "unknown"),
            "votes": [
                {
                    "option_id": vote.option_id,
                    "confidence": vote.confidence,
                    "reasoning": vote.reasoning,
                    "raw_text": str(vote.metadata.get("raw_text", ""))[:300],
                    "model_name": vote.metadata.get("model_name"),
                    "sample_seed": vote.metadata.get("sample_seed"),
                    "option_order": vote.metadata.get("option_order"),
                }
                for vote in votes
            ],
            "judge_raw_text": judge_raw_text,
        }


class RoutedRAGCouncilStrategy(BaseStrategy):
    name = "routed_rag_council"

    def __init__(
        self,
        candidate_llms: list[LocalLLM],
        judge_llm: LocalLLM,
        direct_strategy: BaseStrategy | None = None,
        retrieval_config: RAGConfig | None = None,
        retriever: Callable[[str, RAGConfig, LocalLLM], tuple[str, list[Any], float]] | None = None,
        judge_scope: str = "candidate_only",
        base_seed: int = 500,
        candidate_temperature: float = 0.7,
        candidate_top_p: float = 0.9,
        candidate_max_new_tokens: int = 80,
        judge_max_new_tokens: int = 12,
        max_time_per_call: float | None = 8.0,
        always_judge: bool = True,
        unload_candidates_before_judge: bool = False,
        unload_judge_before_candidates: bool = False,
        candidate_styles: list[str] | None = None,
    ):
        if not candidate_llms:
            raise ValueError("At least one candidate LLM is required")
        if judge_scope not in {"candidate_only", "any_option"}:
            raise ValueError("judge_scope must be 'candidate_only' or 'any_option'")
        self.candidate_llms = list(candidate_llms)
        self.judge_llm = judge_llm
        self.direct_strategy = direct_strategy
        self.cfg = retrieval_config or RAGConfig(num_extra_queries=0)
        self.retriever = retriever or _rag_retrieve
        self.judge_scope = judge_scope
        self.base_seed = base_seed
        self.candidate_temperature = candidate_temperature
        self.candidate_top_p = candidate_top_p
        self.candidate_max_new_tokens = candidate_max_new_tokens
        self.judge_max_new_tokens = judge_max_new_tokens
        self.max_time_per_call = max_time_per_call
        self.always_judge = always_judge
        self.unload_candidates_before_judge = unload_candidates_before_judge
        self.unload_judge_before_candidates = unload_judge_before_candidates
        self.candidate_styles = candidate_styles or []

    def answer(self, question: Question) -> AnswerPrediction:
        if self.unload_judge_before_candidates:
            self._unload_judge_except_candidates()

        decision = route_question(question)
        if decision.reason == "math" and self.direct_strategy is not None:
            prediction = self.direct_strategy.answer(question)
            prediction.metadata["route"] = decision.route
            prediction.metadata["route_reason"] = decision.reason
            prediction.metadata["routed_to"] = getattr(self.direct_strategy, "name", "direct")
            return prediction

        evidence = ""
        evidence_docs: list[Any] = []
        retrieval_seconds = 0.0
        retrieval_error: str | None = None
        use_rag = decision.route == "rag"
        retrieval_query = _retrieval_query(question) if use_rag else ""
        if use_rag:
            try:
                evidence, evidence_docs, retrieval_seconds = self.retriever(
                    retrieval_query,
                    self.cfg,
                    self.candidate_llms[0],
                )
            except Exception as exc:
                retrieval_error = str(exc)
                use_rag = False

        votes = self._candidate_votes(question, evidence if use_rag else None)
        evidence_vote = _evidence_verifier_vote(question, evidence) if use_rag else None
        if evidence_vote is not None:
            votes.append(evidence_vote)
        votes = _filter_unsupported_evidence_votes(question, votes, evidence, use_rag)
        if not votes:
            if self.direct_strategy is not None:
                prediction = self.direct_strategy.answer(question)
                prediction.metadata["route"] = decision.route
                prediction.metadata["route_reason"] = decision.reason
                prediction.metadata["routed_to"] = getattr(self.direct_strategy, "name", "direct")
                prediction.metadata["council_fallback_reason"] = "No valid candidate votes."
                return prediction
            return _council_fallback(question, "No valid candidate votes.", [])

        supported_options = {vote.option_id for vote in votes}
        authoritative_evidence = _authoritative_evidence_option(votes, evidence_vote)
        if authoritative_evidence is not None:
            option_id, evidence_reason = authoritative_evidence
            result = _selected_vote_prediction(question, votes, option_id)
            result.reasoning = "Strong retrieved evidence selected this answer."
            result.metadata.update(self._metadata(votes, None, "evidence_verifier", decision, use_rag, evidence, evidence_docs, retrieval_seconds, retrieval_error, retrieval_query))
            result.metadata["evidence_verifier_reason"] = evidence_reason
            return result

        filtered = _support_filtered_option(votes)
        if filtered is not None:
            option_id, filter_reason = filtered
            result = _selected_vote_prediction(question, votes, option_id)
            result.metadata.update(self._metadata(votes, None, "support_filter", decision, use_rag, evidence, evidence_docs, retrieval_seconds, retrieval_error, retrieval_query))
            result.metadata["support_filter_reason"] = filter_reason
            return result

        majority_option = _majority_option(votes)
        if (
            majority_option is not None
            and not self.always_judge
            and (evidence_vote is None or majority_option == evidence_vote.option_id)
        ):
            result = _selected_vote_prediction(question, votes, majority_option)
            result.metadata.update(self._metadata(votes, None, "majority_vote", decision, use_rag, evidence, evidence_docs, retrieval_seconds, retrieval_error, retrieval_query))
            return result

        if self.unload_candidates_before_judge:
            self._unload_candidates_except_judge()

        judge_raw_text = self.judge_llm.generate(
            build_rag_council_judge_prompt(question, votes, evidence if use_rag else "", self.judge_scope),
            max_new_tokens=self.judge_max_new_tokens,
            do_sample=False,
            seed=self.base_seed + len(votes),
            **self._time_kwargs(),
        )
        judged = parse_answer_prediction(judge_raw_text, question, strategy_name=self.name)
        if not judged.metadata["fallback"] and (
            self.judge_scope == "any_option" or judged.option_id in supported_options
        ):
            judged.metadata.update(self._metadata(votes, judge_raw_text, "judge", decision, use_rag, evidence, evidence_docs, retrieval_seconds, retrieval_error, retrieval_query))
            judged.metadata["judge_novel_choice"] = judged.option_id not in supported_options
            return judged

        result = _weighted_vote(question, votes)
        result.metadata.update(self._metadata(votes, judge_raw_text, "weighted_vote", decision, use_rag, evidence, evidence_docs, retrieval_seconds, retrieval_error, retrieval_query))
        result.metadata["judge_rejected"] = True
        result.metadata["judge_option_id"] = None if judged.metadata["fallback"] else judged.option_id
        return result

    def _candidate_votes(self, question: Question, evidence: str | None) -> list[AnswerPrediction]:
        votes: list[AnswerPrediction] = []
        for index, llm in enumerate(self.candidate_llms):
            style = self.candidate_styles[index] if index < len(self.candidate_styles) else "balanced"
            prompt = build_rag_council_vote_prompt(question, evidence, style)
            raw_text = llm.generate(
                prompt,
                max_new_tokens=self.candidate_max_new_tokens,
                do_sample=True,
                temperature=self.candidate_temperature,
                top_p=self.candidate_top_p,
                seed=self.base_seed + index,
                **self._time_kwargs(),
            )
            vote = parse_answer_prediction(raw_text, question, strategy_name="routed_rag_council_vote")
            if vote.metadata["fallback"]:
                continue
            vote.metadata["model_name"] = getattr(llm, "model_name", "unknown")
            vote.metadata["sample_seed"] = self.base_seed + index
            vote.metadata["candidate_style"] = style
            votes.append(vote)
        return votes

    def _unload_candidates_except_judge(self) -> None:
        judge_roots = {_llm_root_id(self.judge_llm)}
        for llm in self.candidate_llms:
            if _llm_root_id(llm) not in judge_roots:
                unload_strategy(llm)

    def _unload_judge_except_candidates(self) -> None:
        candidate_roots = {_llm_root_id(llm) for llm in self.candidate_llms}
        if _llm_root_id(self.judge_llm) not in candidate_roots:
            unload_strategy(self.judge_llm)

    def _time_kwargs(self) -> dict[str, float]:
        if self.max_time_per_call is None:
            return {}
        return {"max_time": self.max_time_per_call}

    def _metadata(
        self,
        votes: list[AnswerPrediction],
        judge_raw_text: str | None,
        method: str,
        decision: RouteDecision,
        used_rag: bool,
        evidence: str,
        evidence_docs: list[Any],
        retrieval_seconds: float,
        retrieval_error: str | None,
        retrieval_query: str,
    ) -> dict[str, Any]:
        return {
            "strategy": self.name,
            "route": decision.route,
            "route_reason": decision.reason,
            "used_rag": used_rag,
            "retrieval_query": retrieval_query,
            "retrieval_seconds": round(retrieval_seconds, 2),
            "retrieval_error": retrieval_error,
            "num_evidence_chunks": len(evidence_docs),
            "evidence_preview": evidence[:1200] if evidence else "",
            "evidence_sources": _rag_evidence_sources(evidence_docs) if evidence_docs else [],
            "decision_method": method,
            "judge_scope": self.judge_scope,
            "unload_candidates_before_judge": self.unload_candidates_before_judge,
            "unload_judge_before_candidates": self.unload_judge_before_candidates,
            "candidate_styles": self.candidate_styles,
            "candidate_devices": [
                getattr(llm, "device_summary", "unknown") for llm in self.candidate_llms
            ],
            "judge_device": getattr(self.judge_llm, "device_summary", "unknown"),
            "judge_model_name": getattr(self.judge_llm, "model_name", "unknown"),
            "votes": [
                {
                    "option_id": vote.option_id,
                    "confidence": vote.confidence,
                    "reasoning": vote.reasoning,
                    "model_name": vote.metadata.get("model_name"),
                    "sample_seed": vote.metadata.get("sample_seed"),
                    "style": vote.metadata.get("candidate_style"),
                    "raw_text": str(vote.metadata.get("raw_text", ""))[:300],
                }
                for vote in votes
            ],
            "judge_raw_text": judge_raw_text,
        }


def build_prompt(question: Question) -> str:
    options = "\n".join(f"{option.id}) {option.text}" for option in question.options)
    negation_note = (
        "The question contains NOT or EXCEPT. Select the option that does not fit."
        if _has_negation_trap(question.text.lower())
        else "Select the single best option."
    )
    return "\n\n".join(
        [
            "Answer the multiple-choice question.",
            _speech_asr_note(question),
            negation_note,
            "Return exactly: option_id: <number>",
            f"Question: {question.text}",
            options,
        ]
    )


def build_qwen_prompt(question: Question) -> str:
    options = "\n".join(f"{option.id}) {option.text}" for option in question.options)
    negation_note = (
        "Important: NOT/EXCEPT means choose the option that does not fit."
        if _has_negation_trap(question.text.lower())
        else "Choose the best answer."
    )
    return "\n\n".join(
        [
            _speech_asr_note(question),
            negation_note,
            f"Question: {question.text}",
            options,
            "Finish with exactly: option_id: <number>",
        ]
    )


def build_council_vote_prompt(question: Question, option_order: list[Any] | None = None) -> str:
    options = "\n".join(f"{option.id}) {option.text}" for option in (option_order or question.options))
    return "\n\n".join(
        [
            "Pick the best answer. Check words such as NOT and EXCEPT.",
            _speech_asr_note(question),
            "Return JSON with keys option_id, confidence, and reason. Use the listed numeric ID. Keep the reason short.",
            f"Q: {question.text}",
            options,
        ]
    )


def build_judge_prompt(
    question: Question,
    votes: list[AnswerPrediction],
    judge_scope: str = "candidate_only",
) -> str:
    options = "\n".join(f"{option.id}) {option.text}" for option in question.options)
    summaries = "\n".join(
        f"vote {index + 1}: option={vote.option_id}, confidence={vote.confidence}, reason={vote.reasoning or ''}"
        for index, vote in enumerate(votes)
    )
    supported = ", ".join(str(option_id) for option_id in sorted({vote.option_id for vote in votes}))
    scope_text = (
        f"Choose only one proposed option id: {supported}."
        if judge_scope == "candidate_only"
        else "You may choose any listed option."
    )
    return "\n\n".join(
        [
            f"Choose the final answer from the candidates. {scope_text} Return only the option id number.",
            _speech_asr_note(question),
            f"Q: {question.text}",
            options,
            summaries,
            "Answer:",
        ]
    )


def build_rag_council_judge_prompt(
    question: Question,
    votes: list[AnswerPrediction],
    evidence: str,
    judge_scope: str = "candidate_only",
) -> str:
    options = "\n".join(f"{option.id}) {option.text}" for option in question.options)
    summaries = "\n".join(
        f"candidate {index + 1}: option={vote.option_id}, confidence={vote.confidence}, reason={vote.reasoning or ''}"
        for index, vote in enumerate(votes)
    )
    supported = ", ".join(str(option_id) for option_id in sorted({vote.option_id for vote in votes}))
    scope_text = (
        f"Choose only one proposed option id: {supported}."
        if judge_scope == "candidate_only"
        else "You may choose any listed option if the candidates missed it."
    )
    evidence_text = evidence[:1800] if evidence else "No retrieved evidence was used."
    return "\n\n".join(
        [
            "You are the final judge for a multiple-choice quiz.",
            scope_text,
            _speech_asr_note(question),
            "Use candidate agreement, confidence, short reasons, and evidence if it is relevant.",
            "If options overlap, prefer the complete option explicitly supported by evidence over a shorter related title.",
            "Do not choose a candidate whose own reason says the answer is not mentioned or unsupported.",
            "Return exactly: option_id: <number>",
            f"Question: {question.text}",
            options,
            "Retrieved evidence:",
            evidence_text,
            "Candidate answers:",
            summaries,
        ]
    )


def build_rag_council_vote_prompt(
    question: Question,
    evidence: str | None,
    style: str = "balanced",
) -> str:
    options = "\n".join(f"{option.id}) {option.text}" for option in question.options)
    evidence_text = evidence[:1000] if evidence else "No retrieved evidence. Use your knowledge, but lower confidence if uncertain."
    style_instructions = {
        "evidence_checker": (
            "Role: evidence checker. Use retrieved evidence when relevant. "
            "Prefer the exact complete option fully supported by evidence. "
            "When option texts overlap, do not pick a shorter related title unless the evidence supports that shorter title exactly. "
            "If evidence is weak or unrelated, lower confidence."
        ),
        "option_eliminator": (
            "Role: option eliminator. Compare every option against the question. "
            "Reject distractors, watch NOT/EXCEPT wording, overlapping names, dates, exact title length, and causal direction."
        ),
        "balanced": (
            "Role: balanced solver. Combine evidence, question wording, and basic background knowledge."
        ),
    }
    instruction = style_instructions.get(style, style_instructions["balanced"])
    return "\n\n".join(
        [
            instruction,
            _speech_asr_note(question),
            "Return ONLY JSON: {\"option_id\": <number>, \"confidence\": <0-1>, \"reason\": \"short\"}",
            "Evidence:",
            evidence_text,
            f"Question: {question.text}",
            options,
        ]
    )


def _speech_asr_note(question: Question) -> str:
    if question.metadata.get("mode") != "speech":
        return ""
    return (
        "Speech-mode note: question and options are ASR transcripts and may contain errors. "
        "Ignore obvious audio artifacts such as option labels, pfft, laughter, or garbled words. "
        "Infer the likely answer, then choose the closest listed option ID."
    )


def parse_answer_prediction(raw_text: str, question: Question, strategy_name: str = "llm") -> AnswerPrediction:
    payload = _parse_payload(raw_text)
    option_id = _coerce_int(payload.get("option_id"))
    if option_id not in question.valid_option_ids():
        option_id = _match_option_text(raw_text, question)

    fallback = option_id not in question.valid_option_ids()
    if fallback:
        option_id = question.first_option().id

    option = question.require_option(option_id)
    reason = payload.get("reason")
    if reason is not None:
        reason = str(reason).strip()[:300]

    return AnswerPrediction(
        option_id=option.id,
        answer_text=option.text,
        confidence=_clamp_confidence(payload.get("confidence")),
        reasoning=reason,
        metadata={
            "strategy": strategy_name,
            "raw_text": raw_text,
            "fallback": fallback,
            "parsed_payload": payload,
        },
    )


def _majority_option(votes: list[AnswerPrediction]) -> int | None:
    counts: dict[int, int] = {}
    for vote in votes:
        counts[vote.option_id] = counts.get(vote.option_id, 0) + 1
    option_id, count = max(counts.items(), key=lambda item: (item[1], -item[0]))
    return option_id if count > len(votes) / 2 else None


def _filter_unsupported_evidence_votes(
    question: Question,
    votes: list[AnswerPrediction],
    evidence: str,
    use_rag: bool,
) -> list[AnswerPrediction]:
    if not use_rag or not evidence or evidence == "No evidence found.":
        return votes
    filtered: list[AnswerPrediction] = []
    for vote in votes:
        if vote.metadata.get("model_name") == "evidence_verifier":
            filtered.append(vote)
            continue
        style = str(vote.metadata.get("candidate_style", "")).lower()
        reason = f"{vote.reasoning or ''} {vote.metadata.get('raw_text', '')}".lower()
        claims_evidence = style in {"evidence_checker", "option_eliminator"} or any(
            token in reason for token in ("evidence", "source", "text", "explicitly states")
        )
        if claims_evidence and not _option_supported_by_evidence(question, vote.option_id, evidence):
            vote.metadata["rejected_by_evidence_gate"] = True
            vote.metadata["rejection_reason"] = "selected option text was not found in retrieved evidence"
            continue
        filtered.append(vote)
    return filtered


def _option_supported_by_evidence(question: Question, option_id: int, evidence: str) -> bool:
    option = question.require_option(option_id)
    option_text = _normalize_title(option.text)
    evidence_text = _normalize_title(evidence)
    if not option_text:
        return False
    if _phrase_hits(evidence_text, option_text):
        return True
    option_tokens = [token for token in option_text.split() if len(token) > 3]
    if len(option_tokens) >= 3:
        overlap = sum(1 for token in option_tokens if re.search(rf"(?<!\w){re.escape(token)}(?!\w)", evidence_text))
        return overlap / len(option_tokens) >= 0.8
    return False


def _support_filtered_option(votes: list[AnswerPrediction]) -> tuple[int, str] | None:
    if len(votes) < 2:
        return None
    scored = sorted(
        ((vote, _vote_support_score(vote)) for vote in votes),
        key=lambda item: item[1],
        reverse=True,
    )
    best, best_score = scored[0]
    second_score = scored[1][1]
    if best_score >= 0.35 and best_score - second_score >= 0.35:
        return best.option_id, "clearer supported candidate"
    supported = [vote for vote, score in scored if score >= 0.35 and not _vote_says_unsupported(vote)]
    if supported and len({vote.option_id for vote in supported}) == 1:
        return supported[0].option_id, "only supported candidate option"
    return None


def _authoritative_evidence_option(
    votes: list[AnswerPrediction],
    evidence_vote: AnswerPrediction | None,
) -> tuple[int, str] | None:
    if evidence_vote is None or evidence_vote.confidence is None or evidence_vote.confidence < 0.85:
        return None
    scores = evidence_vote.metadata.get("evidence_scores", {})
    if not isinstance(scores, dict):
        return None
    best_score = float(scores.get(evidence_vote.option_id, 0.0))
    other_score = max(
        (float(score) for option_id, score in scores.items() if option_id != evidence_vote.option_id),
        default=0.0,
    )
    if best_score < 3.0 or best_score - other_score < 2.0:
        return None
    opposing = [
        vote for vote in votes
        if vote is not evidence_vote and vote.option_id != evidence_vote.option_id
    ]
    strong_opposing = [vote for vote in opposing if _model_vote_is_well_supported(vote)]
    if len(strong_opposing) >= 2:
        return None
    return evidence_vote.option_id, "strong evidence margin from retrieved sources"


def _model_vote_is_well_supported(vote: AnswerPrediction) -> bool:
    if vote.confidence is None or vote.confidence < 0.85:
        return False
    if _vote_says_unsupported(vote):
        return False
    reason = (vote.reasoning or "").strip()
    if len(reason.split()) < 8:
        return False
    incomplete_endings = ("states", "explicitly states", "because", "while", "option")
    return not reason.lower().rstrip(" .,:;").endswith(incomplete_endings)


def _vote_support_score(vote: AnswerPrediction) -> float:
    confidence = vote.confidence if vote.confidence is not None else 0.5
    score = float(confidence)
    if _vote_says_unsupported(vote):
        score -= 0.6
    return score


def _vote_says_unsupported(vote: AnswerPrediction) -> bool:
    text = f"{vote.reasoning or ''} {vote.metadata.get('raw_text', '')}".lower()
    unsupported_patterns = [
        "not mentioned",
        "not supported",
        "unsupported",
        "no evidence",
        "does not specify",
        "doesn't specify",
        "not in the text",
        "unclear",
    ]
    return any(pattern in text for pattern in unsupported_patterns)


def _selected_vote_prediction(
    question: Question,
    votes: list[AnswerPrediction],
    option_id: int,
) -> AnswerPrediction:
    supporters = [vote for vote in votes if vote.option_id == option_id]
    confidence_values = [vote.confidence for vote in supporters if vote.confidence is not None]
    option = question.require_option(option_id)
    return AnswerPrediction(
        option_id=option.id,
        answer_text=option.text,
        confidence=sum(confidence_values) / len(confidence_values) if confidence_values else None,
        reasoning="Council majority selected this answer.",
        metadata={"strategy": "council", "fallback": False},
    )


def _weighted_vote(question: Question, votes: list[AnswerPrediction]) -> AnswerPrediction:
    scores: dict[int, float] = {}
    counts: dict[int, int] = {}
    for vote in votes:
        weight = vote.confidence if vote.confidence is not None else 0.5
        scores[vote.option_id] = scores.get(vote.option_id, 0.0) + weight
        counts[vote.option_id] = counts.get(vote.option_id, 0) + 1
    option_id = max(scores, key=lambda item: (counts[item], scores[item], -item))
    option = question.require_option(option_id)
    total_score = sum(scores.values())
    return AnswerPrediction(
        option_id=option.id,
        answer_text=option.text,
        confidence=scores[option_id] / total_score if total_score else None,
        reasoning="Confidence-weighted council vote.",
        metadata={"strategy": "council", "fallback": False},
    )


def _council_fallback(question: Question, reason: str, votes: list[AnswerPrediction]) -> AnswerPrediction:
    option = question.first_option()
    return AnswerPrediction(
        option_id=option.id,
        answer_text=option.text,
        confidence=0.0,
        reasoning=reason,
        metadata={
            "strategy": "council",
            "fallback": True,
            "decision_method": "fallback",
            "votes": votes,
        },
    )


def _parse_payload(raw_text: str) -> dict[str, Any]:
    text = raw_text.strip()
    for candidate in [text, *_json_blocks(text)]:
        try:
            data = json.loads(candidate)
            if isinstance(data, dict):
                return data
        except json.JSONDecodeError:
            pass

    payload: dict[str, Any] = {}
    bare_option_match = re.fullmatch(r"\s*(-?\d+)\s*[\.\)]?\s*", text)
    if bare_option_match:
        payload["option_id"] = bare_option_match.group(1)
        return payload
    option_match = re.search(r"(?:option_id|option|answer)\D+(-?\d+)", text, flags=re.IGNORECASE)
    if option_match:
        payload["option_id"] = option_match.group(1)
    confidence_match = re.search(r"confidence\D+([01](?:\.\d+)?)", text, flags=re.IGNORECASE)
    if confidence_match:
        payload["confidence"] = confidence_match.group(1)
    reason_match = re.search(
        r'["\']?(?:reason|justification)["\']?\s*[:=]\s*["\']?([^"\'\n\r}]*)',
        text,
        flags=re.IGNORECASE,
    )
    if reason_match:
        reason = reason_match.group(1).strip(" ,")
        if reason:
            payload["reason"] = reason
    return payload


def _json_blocks(text: str) -> list[str]:
    return re.findall(r"\{.*?\}", text, flags=re.DOTALL)


def _coerce_int(value: Any) -> int | None:
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def _clamp_confidence(value: Any) -> float | None:
    if value is None:
        return None
    try:
        return max(0.0, min(1.0, float(value)))
    except (TypeError, ValueError):
        return None


def _match_option_text(raw_text: str, question: Question) -> int | None:
    lowered = raw_text.lower()
    for option in question.options:
        if option.text.lower() in lowered:
            return option.id
    raw_tokens = set(_words(raw_text)) - _OPTION_STOPWORDS
    best_option_id: int | None = None
    best_score = 0.0
    for option in question.options:
        option_tokens = set(_words(option.text)) - _OPTION_STOPWORDS
        if not option_tokens:
            continue
        overlap = raw_tokens & option_tokens
        if len(overlap) < 2:
            continue
        score = len(overlap) / len(option_tokens)
        if score > best_score:
            best_score = score
            best_option_id = option.id
    if best_score >= 0.5:
        return best_option_id
    return None


_OPTION_STOPWORDS = {
    "a",
    "an",
    "the",
    "of",
    "to",
    "for",
    "and",
    "or",
    "in",
    "on",
    "with",
    "is",
    "are",
    "test",
}


def _tokenize_prompt(processor: Any, prompt: str) -> dict[str, Any]:
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    if hasattr(processor, "apply_chat_template"):
        try:
            return processor.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            )
        except Exception:
            pass
    return processor(_text_chat_prompt(prompt), return_tensors="pt")


def _text_chat_prompt(prompt: str) -> str:
    return f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"


def _require_supported_transformers(version: str) -> None:
    if Version(version) < Version("5.7.0"):
        raise RuntimeError(
            "The local Gemma/Qwen models require transformers>=5.7.0 in this notebook kernel. "
            f"Current transformers version is {version}. Run the dependency install cell "
            "with INSTALL_DEPS=True, then restart the kernel and rerun setup."
        )


def _quantization_kwargs(quantize_4bit: bool) -> dict[str, Any]:
    if not quantize_4bit:
        return {}
    try:
        import bitsandbytes as _bitsandbytes
        del _bitsandbytes
    except ImportError as exc:
        raise RuntimeError(
            "4-bit quantization needs bitsandbytes in this notebook kernel. "
            "Run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup."
        ) from exc

    import torch
    from transformers import BitsAndBytesConfig

    return {
        "quantization_config": BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
    }


def _solve_math_question(question: Question) -> tuple[Any, Any, str] | None:
    text = question.text.replace("$", " ")
    option = _domain_rule_option(question)
    if option is not None:
        return option, option.text, "domain_rule"
    option = _experimental_bias_option(question)
    if option is not None:
        return option, option.text, "experimental_bias"
    option = _statistics_test_option(question)
    if option is not None:
        return option, option.text, "statistics_test"
    value = _bayes_positive_test_value(text)
    method = "bayes_positive_test"
    if value is None:
        value = _sum_of_squares_value(text)
        method = "sum_of_squares"
    if value is None:
        value = _linear_transformation_plane_value(text)
        method = "linear_transformation"
    if value is None:
        value = _linear_interpolation_value(text)
        method = "linear_interpolation"
    if value is None:
        value = _proportion_sample_size_value(text)
        method = "proportion_sample_size"
    if value is None:
        value = _normal_iqr_value(text)
        method = "normal_iqr"
    if value is None:
        value = _correlation_transform_value(text)
        method = "correlation_transform"
    if value is None:
        value = _time_distance_speed_value(text)
        method = "time_distance_speed"
    if value is None:
        value = _combination_value(text)
        method = "combination"
    if value is None:
        value = _homomorphism_count_value(text)
        method = "homomorphism_count"
    if value is None:
        value = _frequency_value(text)
        method = "wave_frequency"
    if value is None:
        value = _arithmetic_value(text)
        method = "arithmetic"
    if value is None:
        return None
    option = _match_numeric_option(question, value)
    if option is None:
        return None
    return option, value, method


def _domain_rule_option(question: Question) -> Any | None:
    lowered = question.text.lower()
    if "prisoner" in lowered and "dilemma" in lowered and "payoff" in lowered:
        candidates = []
        for option in question.options:
            compact = re.sub(r"\s+", "", option.text.upper())
            if all(piece in compact for piece in ("T>R", "R>P", "P>S")):
                candidates.append(option)
        if candidates and ("mutual cooperation" in lowered or "mutual defection" in lowered):
            explicit = [option for option in candidates if "AND" in option.text.upper() or "," in option.text]
            if explicit:
                return max(explicit, key=lambda option: len(option.text))
        if candidates:
            return min(candidates, key=lambda option: len(option.text))
    return None


def _experimental_bias_option(question: Question) -> Any | None:
    lowered = question.text.lower()
    if not any(term in lowered for term in ("bias", "biased", "investigation", "study", "experiment")):
        return None
    if any(term in lowered for term in ("humans", "people", "participants", "population")) and re.search(r"\bmen\b|\bwomen\b", lowered):
        for option in question.options:
            option_text = option.text.lower()
            if re.search(r"\bonly\s+(men|women|males|females)\b", option_text):
                return option
    return None


def _bayes_positive_test_value(text: str) -> float | None:
    lowered = text.lower()
    if "positive" not in lowered or "disease" not in lowered or "test" not in lowered:
        return None
    percentages = [float(value) / 100.0 for value in re.findall(r"(\d+(?:\.\d+)?)\s*%", lowered)]
    if len(percentages) < 3:
        return None
    prevalence, sensitivity, false_positive = percentages[:3]
    denominator = prevalence * sensitivity + (1.0 - prevalence) * false_positive
    if denominator <= 0:
        return None
    return prevalence * sensitivity / denominator


def _statistics_test_option(question: Question) -> Any | None:
    lowered = question.text.lower()
    if not any(token in lowered for token in ("significance test", "correct test", "t-test", "experiment")):
        return None
    paired_clues = (
        "same volunteer",
        "same subject",
        "same person",
        "one side",
        "other side",
        "before and after",
        "difference in",
        "paired",
        "matched",
    )
    if any(clue in lowered for clue in paired_clues):
        return _option_with_terms(question, {"matched", "pairs", "paired"})
    two_sample_clues = ("two independent", "independent samples", "two groups")
    if any(clue in lowered for clue in two_sample_clues):
        return _option_with_terms(question, {"two", "sample", "t", "test"})
    return None


def _linear_transformation_plane_value(text: str) -> int | float | None:
    lowered = text.lower()
    if "linear transformation" not in lowered:
        return None
    pairs = re.findall(
        r"f\s*\(\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)\s*\)\s*=\s*(-?\d+(?:\.\d+)?)",
        lowered,
    )
    target = re.search(
        r"then\s+f\s*\(\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)\s*\)",
        lowered,
    )
    if len(pairs) < 2 or target is None:
        return None
    x1, y1, v1 = (float(value) for value in pairs[0])
    x2, y2, v2 = (float(value) for value in pairs[1])
    tx, ty = (float(value) for value in target.groups())
    determinant = x1 * y2 - x2 * y1
    if abs(determinant) < 1e-12:
        return None
    a = (v1 * y2 - v2 * y1) / determinant
    b = (x1 * v2 - x2 * v1) / determinant
    value = a * tx + b * ty
    return int(round(value)) if abs(value - round(value)) < 1e-9 else value


def _linear_interpolation_value(text: str) -> int | float | None:
    lowered = text.lower()
    if "linearly" not in lowered and "linear" not in lowered:
        return None
    pairs = re.findall(
        r"(?:in|by)\s+(\d{4}).{0,80}?(?:there\s+(?:were|was)\s+)?([\d,]+)\s+(?:cases|reports|people|students|items)",
        lowered,
    )
    if len(pairs) < 2:
        return None
    years = [int(year) for year in re.findall(r"\b(1[89]\d{2}|20\d{2})\b", lowered)]
    known_years = {int(year) for year, _ in pairs[:2]}
    target_years = [year for year in years if year not in known_years]
    if not target_years:
        return None
    x1, y1 = int(pairs[0][0]), float(pairs[0][1].replace(",", ""))
    x2, y2 = int(pairs[1][0]), float(pairs[1][1].replace(",", ""))
    target = target_years[-1]
    if x1 == x2:
        return None
    value = y1 + ((y2 - y1) / (x2 - x1)) * (target - x1)
    return int(round(value)) if abs(value - round(value)) < 1e-6 else value


def _proportion_sample_size_value(text: str) -> int | None:
    lowered = text.lower()
    if "sample size" not in lowered or "confidence interval" not in lowered:
        return None
    if "margin of error" not in lowered and "margin" not in lowered:
        return None
    percentages = [float(value) for value in re.findall(r"(\d+(?:\.\d+)?)\s*%", lowered)]
    if not percentages:
        return None
    confidence = next((value for value in percentages if value in {90.0, 95.0, 99.0}), 95.0)
    margin_candidates = [value for value in percentages if value < 50.0 and value != confidence]
    if not margin_candidates:
        return None
    margin = min(margin_candidates) / 100.0
    z_scores = {90.0: 1.645, 95.0: 1.96, 99.0: 2.576}
    z = z_scores.get(confidence, 1.96)
    if margin <= 0:
        return None
    return math.ceil((z * z * 0.25) / (margin * margin))


def _normal_iqr_value(text: str) -> float | None:
    lowered = text.lower()
    if "normally distributed" not in lowered or "interquartile range" not in lowered:
        return None
    match = re.search(r"standard deviation(?:\s+of|\s+is)?\s+(\d+(?:\.\d+)?)", lowered)
    if not match:
        return None
    return round(1.35 * float(match.group(1)), 2)


def _correlation_transform_value(text: str) -> float | None:
    lowered = text.lower()
    if "correlation" not in lowered:
        return None
    if not any(token in lowered for token in ("added to all values", "doubled", "interchanged", "swapped")):
        return None
    match = re.search(r"correlation(?:\s+between\s+two\s+variables)?\s+(?:is|=)\s*(-?\d+(?:\.\d+)?)", lowered)
    if not match:
        return None
    value = float(match.group(1))
    return int(value) if value.is_integer() else value


def _time_distance_speed_value(text: str) -> float | None:
    lowered = text.lower()
    if not all(token in lowered for token in ("time", "distance", "speed")):
        return None
    distance_match = re.search(r"distance(?:\s+of)?\s+(\d+(?:\.\d+)?)\s*(?:meters?|m)\b", lowered)
    speed_match = re.search(r"speed(?:\s+of)?\s+(\d+(?:\.\d+)?)\s*(?:meters?|m)\s+per\s+second", lowered)
    if not distance_match or not speed_match:
        return None
    speed = float(speed_match.group(1))
    if speed <= 0:
        return None
    value = float(distance_match.group(1)) / speed
    return int(round(value)) if abs(value - round(value)) < 1e-6 else value


def _option_with_terms(question: Question, terms: set[str]) -> Any | None:
    best = None
    best_score = 0
    for option in question.options:
        tokens = set(_words(option.text))
        score = len(tokens & terms)
        if score > best_score:
            best = option
            best_score = score
    return best if best_score >= 2 else None


def _combination_value(text: str) -> int | None:
    patterns = [
        r"\\d?binom\s*\{\s*(\d+)\s*\}\s*\{\s*(\d+)\s*\}",
        r"\b[Cc]\s*\(\s*(\d+)\s*,\s*(\d+)\s*\)",
        r"\b(\d+)\s+choose\s+(\d+)\b",
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if not match:
            continue
        n, k = int(match.group(1)), int(match.group(2))
        if 0 <= k <= n:
            return math.comb(n, k)
    return None


def _homomorphism_count_value(text: str) -> int | None:
    lowered = text.lower().replace("\\mathbb", "")
    match = re.search(r"homomorphisms?.*?\bz\b.*?\bz[_\s-]?(\d+)\b", lowered)
    if match:
        return int(match.group(1))
    return None


def _frequency_value(text: str) -> float | None:
    lowered = text.lower()
    if "frequency" not in lowered or "wavelength" not in lowered or "speed" not in lowered:
        return None
    wavelength_match = re.search(r"wavelength(?:\s+of)?\s+(\d+(?:\.\d+)?)\s*m\b", lowered)
    speed_match = re.search(r"speed(?:\s+of\s+sound)?\s+is\s+(\d+(?:\.\d+)?)\s*m/s\b", lowered)
    if not wavelength_match or not speed_match:
        return None
    wavelength = float(wavelength_match.group(1))
    speed = float(speed_match.group(1))
    if wavelength <= 0:
        return None
    value = speed / wavelength
    return int(round(value)) if abs(value - round(value)) < 0.6 else value


def _sum_of_squares_value(text: str) -> int | None:
    lowered = text.lower()
    if "^2" not in lowered:
        return None
    target = re.split(r"value of|calculate|what is", lowered, maxsplit=1)
    segment = target[-1] if target else lowered
    nums = [int(value) for value in re.findall(r"(\d+)\s*\^\s*2", segment)]
    if len(nums) < 2:
        return None
    if any(token in segment for token in ("cdots", "...", "…")):
        start, end = nums[0], nums[-1]
        if start > end:
            return None
        return sum(number * number for number in range(start, end + 1))
    return sum(number * number for number in nums)


def _arithmetic_value(text: str) -> int | float | None:
    text = re.sub(r"\b\d{4}[-/]\d{1,2}[-/]\d{1,2}\b", " ", text)
    matches = re.findall(r"(?<!\w)(\d+(?:\.\d+)?(?:\s*[\+\*/]\s*\d+(?:\.\d+)?)+)(?!\w)", text)
    if not matches:
        matches = re.findall(r"(?<!\w)(\d+(?:\.\d+)?\s+-\s+\d+(?:\.\d+)?)(?!\w)", text)
    if not matches:
        return None
    expression = matches[-1]
    try:
        value = eval(expression, {"__builtins__": None}, {})
    except Exception:
        return None
    if isinstance(value, float) and value.is_integer():
        return int(value)
    return value if isinstance(value, (int, float)) else None


def _match_numeric_option(question: Question, value: int | float) -> Any | None:
    target = float(value)
    best_option = None
    best_diff = float("inf")
    for option in question.options:
        cleaned = option.text.replace(",", "")
        numbers = re.findall(r"-?\d+(?:\.\d+)?", cleaned)
        for number in numbers:
            number_value = float(number)
            candidates = [number_value]
            if "%" in cleaned or (0 <= target <= 1 and abs(number_value) > 1):
                candidates.append(number_value / 100.0)
            for candidate in candidates:
                diff = abs(candidate - target)
                if diff < best_diff:
                    best_option = option
                    best_diff = diff
    tolerance = max(1e-6, 0.01 * max(1.0, abs(target)))
    return best_option if best_option is not None and best_diff <= tolerance else None


def _normalize_title(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", text.lower()).strip()


def _evidence_verifier_vote(question: Question, evidence: str) -> AnswerPrediction | None:
    if not evidence or evidence == "No evidence found.":
        return None
    scored = _evidence_option_scores(question, evidence)
    if not scored:
        return None
    ranked = sorted(scored.items(), key=lambda item: (item[1], len(question.require_option(item[0]).text)), reverse=True)
    best_option_id, best_score = ranked[0]
    second_score = ranked[1][1] if len(ranked) > 1 else 0.0
    if best_score < 3.0 or best_score - second_score < 2.0:
        return None
    option = question.require_option(best_option_id)
    confidence = min(0.9, 0.55 + (best_score - second_score) / 10)
    return AnswerPrediction(
        option_id=option.id,
        answer_text=option.text,
        confidence=confidence,
        reasoning="Retrieved evidence explicitly supports this option more than the alternatives.",
        metadata={
            "strategy": "evidence_verifier",
            "fallback": False,
            "model_name": "evidence_verifier",
            "candidate_style": "evidence_verifier",
            "evidence_scores": scored,
        },
    )


def _evidence_option_scores(question: Question, evidence: str) -> dict[int, float]:
    evidence_text = _normalize_title(evidence)
    title_text = _normalize_title(" ".join(_evidence_titles(evidence)))
    scores: dict[int, float] = {}
    for option in question.options:
        option_text = _normalize_title(option.text)
        if not option_text:
            continue
        token_count = len(option_text.split())
        text_hits = _phrase_hits(evidence_text, option_text)
        title_hits = _phrase_hits(title_text, option_text)
        if text_hits or title_hits:
            token_weight = max(1.0, float(token_count))
            scores[option.id] = text_hits * token_weight + title_hits * token_weight * 2.0
    return scores


def _evidence_titles(evidence: str) -> list[str]:
    titles: list[str] = []
    for line in evidence.splitlines():
        match = re.match(r"\[\d+\]\s+(.+)", line.strip())
        if match:
            titles.append(match.group(1))
    return titles


def _phrase_hits(text: str, phrase: str) -> int:
    return len(re.findall(rf"(?<!\w){re.escape(phrase)}(?!\w)", text))


def _words(text: str) -> list[str]:
    return re.findall(r"[A-Za-z0-9]+", text.lower())


def _looks_like_math(text: str) -> bool:
    text_without_dates = re.sub(r"\b\d{4}[-/]\d{1,2}[-/]\d{1,2}\b", " ", text)
    math_words = {
        "calculate",
        "calculation",
        "sum",
        "difference",
        "product",
        "divide",
        "multiply",
        "plus",
        "minus",
        "percent",
        "percentage",
        "probability",
        "average",
        "binomial",
        "coefficient",
        "combination",
        "combinations",
        "frequency",
        "wavelength",
        "correlation",
        "distance",
        "speed",
        "homomorphism",
        "homomorphisms",
        "significance",
        "experiment",
        "t-test",
        "linear",
        "transformation",
        "confidence",
        "interval",
        "margin",
        "sample",
        "normally",
        "distributed",
        "standard",
        "deviation",
        "interquartile",
    }
    if re.search(r"\\d?binom\s*\{\s*\d+\s*\}\s*\{\s*\d+\s*\}", text_without_dates):
        return True
    if re.search(r"\b[Cc]\s*\(\s*\d+\s*,\s*\d+\s*\)", text_without_dates):
        return True
    if re.search(r"\b\d+\s+choose\s+\d+\b", text_without_dates):
        return True
    if re.search(r"\b(significance test|t-test|homomorphisms?)\b", text_without_dates):
        return True
    if re.search(r"\d+\s*[\+\*/]\s*\d+", text_without_dates):
        return True
    if re.search(r"\d+\s+-\s+\d+", text_without_dates):
        return True
    return any(word in _words(text) for word in math_words)


def _has_negation_trap(text: str) -> bool:
    lowered = text.lower()
    if re.search(r"\b(except|least|incorrect|false)\b", lowered):
        return True
    if re.search(r"\bnot\s+(written|composed|created|included|true|correct|associated|mentioned|supported|used)\b", lowered):
        return True
    return bool(re.search(r"\bnone\s+of\s+the\s+above\b", lowered))


def _looks_factual(text: str) -> bool:
    factual_words = {
        "which",
        "who",
        "when",
        "where",
        "what",
        "according",
        "song",
        "album",
        "film",
        "movie",
        "actor",
        "actress",
        "artist",
        "history",
        "born",
        "written",
        "known",
        "describes",
        "influence",
    }
    return any(word in _words(text) for word in factual_words)


def _looks_recent_or_report(text: str) -> bool:
    if re.search(r"\b(?:19|20)\d{2}[-/]\d{1,2}[-/]\d{1,2}\b", text):
        return True
    report_words = {"report", "according", "news", "current", "recent", "coach", "stated"}
    return any(word in _words(text) for word in report_words)


def _clear_torch_memory() -> None:
    try:
        import gc
        import torch

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass


def unload_rag_runtime() -> None:
    global _RAG_EMBED_MODEL, _RAG_CROSS_ENCODER, _RAG_SPLITTER, _RAG_RUNTIME_READY
    _RAG_EMBED_MODEL = None
    _RAG_CROSS_ENCODER = None
    _RAG_SPLITTER = None
    _RAG_RUNTIME_READY = False
    _clear_torch_memory()


def unload_strategy(strategy: BaseStrategy | None) -> None:
    seen: set[int] = set()

    def unload_item(item: Any) -> None:
        if item is None:
            return
        marker = id(item)
        if marker in seen:
            return
        seen.add(marker)
        if hasattr(item, "unload"):
            item.unload()
            return
        if isinstance(item, LocalLLMVariant):
            unload_item(item.llm)
            return
        if isinstance(item, GemmaStrategy) or isinstance(item, QwenStrategy) or isinstance(item, RAGStrategy):
            unload_item(getattr(item, "llm", None))
        if isinstance(item, CalculatorStrategy):
            unload_item(item.fallback_strategy)
        if isinstance(item, RoutedStrategy):
            unload_item(item.direct_strategy)
            unload_item(item.rag_strategy)
            unload_item(item.fallback_strategy)
            unload_item(item.low_confidence_strategy)
        if isinstance(item, CouncilStrategy):
            for llm in item.candidate_llms:
                unload_item(llm)
            unload_item(item.judge_llm)
        if isinstance(item, RoutedRAGCouncilStrategy):
            unload_item(item.direct_strategy)
            for llm in item.candidate_llms:
                unload_item(llm)
            unload_item(item.judge_llm)
        if isinstance(item, LangChainAgenticStrategy):
            unload_item(item.raw_llm)

    unload_item(strategy)
    _clear_torch_memory()


def _llm_root_id(llm: Any) -> int:
    while isinstance(llm, LocalLLMVariant):
        llm = llm.llm
    return id(llm)


def web_search_tool(query: str) -> str:
    """
    Tool for modern pop culture, movies, music, recent events (up to current year 2026), 
    awards, or current news. Input must be a short, concise web search query string.
    """
    import urllib.request
    import urllib.parse
    import json
    try:
        clean_query = urllib.parse.quote(query.strip())
        url = f"https://api.duckduckgo.com/?q={clean_query}&format=json&no_html=1&skip_disambig=1"
        
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=4) as response:
            data = json.loads(response.read().decode())
            
        abstract = data.get("AbstractText", "")
        if abstract:
            return f"Web Search Context: {abstract}"
            
        related = data.get("RelatedTopics", [])
        if related and "Text" in related[0]:
            return f"Web Search Context: {related[0]['Text']}"
            
        return "No direct web results found. Try simplifying the query."
    except Exception as e:
        return f"Web search failed: {str(e)}"

def calculator_tool(expression: str) -> float:
    """
    Computes mathematical operations, this tool does.
    An argument it takes, a basic Python math string like '2 + 2' or '1.95 ** 10' it must be.
    """
    clean_expr = re.sub(r'[^0-9\+\-\*\/\(\)\.\s]', '', expression)
    return float(eval(clean_expr, {"__builtins__": None}, {}))

def wikipedia_tool(search_term: str) -> str:
    """
    Historical facts, biographies or general knowledge, this tool finds.
    A single entity, historical person, or academic concept as search_term string, it requires.
    """
    import urllib.request
    import urllib.parse
    import json
    from bs4 import BeautifulSoup
    try:
        query = urllib.parse.quote(search_term.strip())
        url = f"https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch={query}&format=json"
        req = urllib.request.Request(url, headers={'User-Agent': 'PoliMillionaireAgent/1.0'})
        
        with urllib.request.urlopen(req, timeout=4) as response:
            data = json.loads(response.read().decode())
            
        results = data.get("query", {}).get("search", [])
        if not results:
            return "No information found in Wikipedia."
            
        snippet = BeautifulSoup(results[0].get("snippet", ""), "html.parser").get_text()
        return f"Fact context found: {snippet}"
    except Exception as e:
        return f"Error during search: {str(e)}"



class LangChainAgenticStrategy(BaseStrategy):
    name = "langchain_agent"

    def __init__(self, raw_llm: LocalLLM, fallback_strategy: BaseStrategy | None = None):
        try:
            from langchain_core.tools import tool
        except ImportError as exc:
            raise RuntimeError(
                "LangChainAgenticStrategy needs langchain-core installed. "
                "Use the normal Gemma/Qwen/RAG strategies, or install the optional LangChain dependency."
            ) from exc
        self.raw_llm = raw_llm
        self.fallback = fallback_strategy or HeuristicStrategy()
        self.tools = [tool(calculator_tool), tool(wikipedia_tool), tool(web_search_tool)]
        self._setup_agent_prompts()

    def _setup_agent_prompts(self):
        from langchain_core.prompts import ChatPromptTemplate
        from langchain_core.tools import render_text_description

        rendered_tools = render_text_description(self.tools)
        
        system_routing = f"""
You are an advanced orchestrator for the quiz game 'Who wants to be a PoliMillionaire?'.
Your job is to decide if you need an external tool to answer the question accurately.

Here are the names and descriptions for each tool available:
{rendered_tools}

STRICT SELECTION CRITERIA:
1. If the question requires math, counting, formulas, or arithmetic operations, you MUST choose 'calculator_tool'.
2. If the question is about established history, global geography, science, or older biographies, you MUST choose 'wikipedia_tool'.
3. If the question is about modern pop culture, movies, actors, music, awards, current news, or events up to the current year 2026, you MUST choose 'web_search_tool'.
4. If you DO NOT need a tool to answer the question, you MUST set 'name': "none" and 'arguments': {{"query": ""}}.

The response format MUST be exactly a valid JSON block like this:
{{"name": "tool_name", "arguments": {{"query": "search query text here"}}}}
"""
        self.routing_prompt = ChatPromptTemplate.from_messages([
            ("system", system_routing),
            ("user", "Question: {{input}}\nOptions: {{options_text}}")
        ], template_format="jinja2")

        system_answering = """
You are a brilliant contestant playing 'Who wants to be a PoliMillionaire?'.
Additional verified context from your tools: {{context}}

Analyze the question and select the single best option ID.
You MUST return your response ONLY as a JSON blob matching this structure:
{"option_id": 0, "confidence": 0.95, "reasoning": "Your step-by-step logic here"}
"""
        self.answer_prompt = ChatPromptTemplate.from_messages([
            ("system", system_answering),
            ("user", "Question: {{input}}\nOptions:\n{{options_text}}")
        ], template_format="jinja2")

    

    def _invoke_tool(self, tool_name: str, tool_args: dict) -> str:
        if tool_name == "none" or not tool_name:
            return "No additional context needed."
            
        tool_map = {t.name: t for t in self.tools}
        if tool_name in tool_map:
            try:
                actual_args = tool_args
                if tool_name in ["wikipedia_tool", "web_search_tool"]:
                    extracted_text = tool_args.get("query") or tool_args.get("search_term") or tool_args.get("expression")
                    if not extracted_text and isinstance(tool_args, dict) and tool_args:
                        extracted_text = list(tool_args.values())[0]
                    
                    if tool_name == "wikipedia_tool":
                        actual_args = {"search_term": str(extracted_text or "")}
                    else:
                        actual_args = {"query": str(extracted_text or "")}
                        
                elif tool_name == "calculator_tool":
                    extracted_expr = tool_args.get("expression") or tool_args.get("query")
                    if not extracted_expr and isinstance(tool_args, dict) and tool_args:
                        extracted_expr = list(tool_args.values())[0]
                    actual_args = {"expression": str(extracted_expr or "2+2")}

                result = tool_map[tool_name].invoke(actual_args)
                return str(result)
            except Exception as e:
                return f"Tool execution failed: {str(e)}"
        return "Unknown tool requested."

    def answer(self, question: Question) -> AnswerPrediction:
        try:
            import traceback  # <-- Para ver la línea exacta del error
            options_str = "\n".join([f"ID {o.id}: {o.text}" for o in question.options])
            
            formatted_routing_prompt = self.routing_prompt.format(
                input=question.text,
                options_text=options_str
            )
            
            if hasattr(self.raw_llm, 'generate'):
                routing_output = self.raw_llm.generate(formatted_routing_prompt, max_new_tokens=512, max_time=30.0)
            else:
                routing_output = self.raw_llm.invoke(formatted_routing_prompt)
            
            routing_text = getattr(routing_output, "text_content", str(routing_output))
            
            print(f"[AGENT DEBUG] Phase 1: Routing raw output: ---\n{routing_text}\n-------------------------------------------------")
            
            try:
                match_routing = re.search(r'\{.*\}', routing_text, re.DOTALL)
                if match_routing:
                    routing_json = json.loads(match_routing.group(0))
                    tool_name = routing_json.get("name", "none")
                    tool_arguments = routing_json.get("arguments", {})
            except Exception as json_err:
                print(f"[AGENT DEBUG] Error parsing tools JSON. {json_err}")
            
            context_data = self._invoke_tool(tool_name, tool_arguments)
            print(f"[Agent Debug] Phase 2: Chosen Tool [{tool_name}] | Retrieved context: {context_data[:100]}...")
            
            formatted_answer_prompt = self.answer_prompt.format(
                context=context_data,
                input=question.text,
                options_text=options_str
            )
            
            if hasattr(self.raw_llm, 'generate'):
                final_output = self.raw_llm.generate(formatted_answer_prompt, max_new_tokens=512, max_time=30.0)
            else:
                final_output = self.raw_llm.invoke(formatted_answer_prompt)
                
            final_text = getattr(final_output, "text_content", str(final_output))
            
            print(f"[DEBUG AGENTE] Phase 3: Final raw output ---\n{final_text}\n-------------------------------------------------")
            
            prediction_data = self._robust_parse_final_answer(final_text, question)
            if prediction_data:
                prediction_data.metadata.update({
                    "strategy": self.name,
                    "used_tool": tool_name,
                    "tool_arguments": str(tool_arguments),
                    "tool_output": context_data,
                    "fallback": False
                })
                return prediction_data
            else:
                print("❌ [Agent Debug] Phase 4: '_robust_parse_final_answer' returned none. The text does not contain any processable format.")
                
        except Exception as e:
            print(f"[AGENT DEBUG] Internal crash in strategy")
            traceback.print_exc()
            
        fallback_pred = self.fallback.answer(question)
        fallback_pred.metadata["fallback"] = True
        fallback_pred.metadata["fallback_reason"] = "Pipeline exception occurred"
        return fallback_pred

    def _robust_parse_final_answer(self, text: str, question: Question) -> AnswerPrediction | None:
        try:
            match = re.search(r'\{.*\}', text, re.DOTALL)
            if match:
                clean_json_str = match.group(0)
                data = json.loads(clean_json_str)
                
                raw_id = data.get("option_id")
                if raw_id is not None:
                    selected_id = int(raw_id)
                    chosen_option = next((o for o in question.options if o.id == selected_id), question.options[0])
                    
                    return AnswerPrediction(
                        option_id=chosen_option.id,
                        answer_text=chosen_option.text,
                        confidence=float(data.get("confidence", 0.85)),
                        reasoning=str(data.get("reasoning", "Parsed via robust JSON engine.")),
                        metadata={}
                    )
        except Exception as parse_exc:
            print(f"[{self.name}] Regex JSON parser missed, fallback to heuristic extraction: {parse_exc}")

        for token in text.split():
            clean_token = re.sub(r'[^0-9]', '', token)
            if clean_token.isdigit():
                val = int(clean_token)
                for opt in question.options:
                    if opt.id == val:
                        return AnswerPrediction(
                            option_id=opt.id,
                            answer_text=opt.text,
                            confidence=0.5,
                            reasoning=f"Extracted direct integer reference '{val}' from corrupted output.",
                            metadata={}
                        )
        return None




_rag_log = logging.getLogger(f"{__name__}.rag")

_RAG_SPLITTER: Any = None

_RAG_EMBED_MODEL: Any = None

_RAG_CROSS_ENCODER: Any = None
_RAG_RUNTIME_READY = False
HAS_NEWSPAPER = False
NewspaperArticle: Any = None


def _ensure_rag_runtime() -> None:
    global _RAG_SPLITTER, _RAG_EMBED_MODEL, _RAG_CROSS_ENCODER
    global _RAG_RUNTIME_READY, HAS_NEWSPAPER, NewspaperArticle
    global httpx, np, trafilatura, DDGS, BM25Retriever, Document

    if _RAG_RUNTIME_READY:
        return

    try:
        import httpx as _httpx
        import nest_asyncio
        import numpy as _np
        import trafilatura as _trafilatura
        from ddgs import DDGS as _DDGS
        from langchain_community.retrievers import BM25Retriever as _BM25Retriever
        from langchain_core.documents import Document as _Document
        from langchain_text_splitters import RecursiveCharacterTextSplitter
        from sentence_transformers import CrossEncoder, SentenceTransformer
    except ImportError as exc:
        raise RuntimeError(
            "RAG needs optional packages: ddgs, httpx, trafilatura, langchain-community, "
            "langchain-text-splitters, sentence-transformers, and rank_bm25."
        ) from exc

    try:
        from newspaper import Article as _NewspaperArticle

        NewspaperArticle = _NewspaperArticle
        HAS_NEWSPAPER = True
    except ImportError:
        NewspaperArticle = None
        HAS_NEWSPAPER = False

    nest_asyncio.apply()
    httpx = _httpx
    np = _np
    trafilatura = _trafilatura
    DDGS = _DDGS
    BM25Retriever = _BM25Retriever
    Document = _Document

    _RAG_SPLITTER = RecursiveCharacterTextSplitter(
        chunk_size=450,
        chunk_overlap=80,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    _rag_log.info("RAG: loading dense encoder on CPU")
    _RAG_EMBED_MODEL = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cpu")
    _rag_log.info("RAG: loading cross-encoder on CPU")
    _RAG_CROSS_ENCODER = CrossEncoder("BAAI/bge-reranker-base", device="cpu")
    _RAG_RUNTIME_READY = True




@dataclass
class RAGConfig:
    max_ddg_results: int = 4
    timeout_ddg: int = 5
    num_extra_queries: int = 1

    fetch_timeout: float = 4.0
    max_concurrent_fetches: int = 4
    max_text_chars: int = 12_000
    total_fetch_budget: float = 5.0      # hard wall-clock cap for the fetch phase

    bm25_top_k: int = 6
    dense_top_k: int = 6
    rrf_k: int = 60                      # standard RRF constant
    diversity_max_per_url: int = 2

    answer_max_new_tokens: int = 96

    final_top_k: int = 3



_RAG_EXPANSION_PROMPT = """\
Generate {n} alternative search queries for the question below.
Rules: each must be semantically distinct, under 12 words, no numbering.
Output ONLY the queries, one per line.

Question: {query}"""

_RAG_ANSWER_PROMPT = """\
Answer this multiple-choice question using only the evidence below when it is relevant.
If the question contains NOT or EXCEPT, choose the option that is not supported or does not fit.
Options may overlap. Choose the exact listed option fully supported by the evidence, not a shorter related option.
If the question asks what caused or led to something, choose the earlier cause, not the later action.
Return ONLY a JSON object with keys: option_id, confidence, reason.

Evidence:
<<<EVIDENCE>>>

Question: <<<QUESTION>>>
<<<OPTIONS>>>"""


def build_rag_prompt(question: Question, evidence: str) -> str:
    options = "\n".join(f"{o.id}) {o.text}" for o in question.options)
    return (
        _RAG_ANSWER_PROMPT
        .replace("<<<EVIDENCE>>>", evidence)
        .replace("<<<QUESTION>>>", question.text)
        .replace("<<<OPTIONS>>>", options)
    )




def _expand_query_rag(query: str, cfg: RAGConfig, llm: LocalLLM) -> list[str]:
    if cfg.num_extra_queries == 0:
        return [query]
    prompt = (
        _RAG_EXPANSION_PROMPT
        .replace("{n}", str(cfg.num_extra_queries))
        .replace("{query}", query)
    )
    try:
        raw = llm.generate(prompt, max_new_tokens=80, do_sample=False)
        variants = [
            line.strip()
            for line in raw.strip().splitlines()
            if line.strip() and line.strip().lower() != query.lower()
        ][: cfg.num_extra_queries]
    except Exception as exc:
        _rag_log.warning("RAG query expansion failed: %s", exc)
        variants = []
    all_queries = [query] + variants
    _rag_log.info("RAG expanded queries: %s", all_queries)
    return all_queries




def _rag_search_all(queries: list[str], cfg: RAGConfig) -> list[dict]:
    seen: set[str] = set()
    merged: list[dict] = []
    for q in queries:
        try:
            with DDGS(timeout=cfg.timeout_ddg) as ddgs:
                results = list(ddgs.text(q, max_results=cfg.max_ddg_results))
        except Exception as exc:
            _rag_log.warning("RAG DDG search failed for %r: %s", q, exc)
            continue
        for r in results:
            url = (r.get("href") or "").strip().lower().rstrip("/")
            if url and url not in seen:
                seen.add(url)
                merged.append(r)
    _rag_log.info("RAG: %d unique URLs after search", len(merged))
    return merged



_RAG_HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; RAGBot/1.0)"}
_RAG_BLOCKED_EXTENSIONS = {".pdf", ".doc", ".docx", ".ppt", ".pptx", ".xls", ".xlsx"}
_RAG_BLOCKED_DOMAINS = {
    "twitter.com", "x.com", "instagram.com",
    "facebook.com", "tiktok.com", "reddit.com",
    "quora.com", "youtube.com", "youtu.be", "pinterest.com",
    "linkedin.com", "threads.net", "tumblr.com",
    "brainly.com", "chegg.com", "coursehero.com", "quizlet.com",
    "studocu.com", "answers.com",
}


def _rag_blocked_domain(url: str) -> bool:
    from urllib.parse import urlparse
    parsed = urlparse(url.lower())
    return any(parsed.netloc == d or parsed.netloc.endswith("." + d) for d in _RAG_BLOCKED_DOMAINS)


def _rag_is_fetchable(url: str) -> bool:
    from urllib.parse import urlparse
    parsed = urlparse(url.lower())
    if _rag_blocked_domain(url):
        return False
    if any(parsed.path.endswith(ext) for ext in _RAG_BLOCKED_EXTENSIONS):
        return False
    return True


def _rag_extract_text(html: str) -> str:
    text = trafilatura.extract(html, include_comments=False, include_tables=False, favor_recall=True)
    if text and len(text) >= 200:
        return text
    if HAS_NEWSPAPER:
        try:
            article = NewspaperArticle(url="")
            article.set_html(html)
            article.parse()
            if article.text and len(article.text) >= 200:
                return article.text
        except Exception:
            pass
    return ""


async def _rag_fetch_one(
    client: httpx.AsyncClient,
    semaphore: asyncio.Semaphore,
    result: dict,
    cfg: RAGConfig,
) -> list[Document]:
    url: str = result.get("href", "")
    title: str = result.get("title", "No title")
    snippet: str = result.get("body", "")

    def _snippet_doc() -> list[Document]:
        return (
            [Document(page_content=snippet, metadata={"title": title, "url": url, "snippet": snippet, "chunk_id": 0})]
            if snippet else []
        )

    if not url:
        return _snippet_doc()
    if _rag_blocked_domain(url):
        return []
    if not _rag_is_fetchable(url):
        return _snippet_doc()

    async with semaphore:
        try:
            resp = await client.get(url, headers=_RAG_HEADERS, timeout=cfg.fetch_timeout)
            resp.raise_for_status()
            html = resp.text
        except Exception as exc:
            _rag_log.debug("RAG fetch failed for %s: %s", url, exc)
            return _snippet_doc()

    text = _rag_extract_text(html) or snippet
    text = " ".join(text.split())[: cfg.max_text_chars]
    chunks = _RAG_SPLITTER.split_text(text)
    return [
        Document(page_content=chunk, metadata={"title": title, "url": url, "snippet": snippet, "chunk_id": i})
        for i, chunk in enumerate(chunks)
    ]


async def _rag_fetch_all_async(results: list[dict], cfg: RAGConfig) -> list[Document]:
    semaphore = asyncio.Semaphore(cfg.max_concurrent_fetches)
    async with httpx.AsyncClient(follow_redirects=True) as client:
        tasks = [asyncio.ensure_future(_rag_fetch_one(client, semaphore, r, cfg)) for r in results]
        done, pending = await asyncio.wait(tasks, timeout=cfg.total_fetch_budget)
        if pending:
            _rag_log.info("RAG: cancelling %d slow fetches (budget %.1fs exceeded)", len(pending), cfg.total_fetch_budget)
            for t in pending:
                t.cancel()

    docs: list[Document] = []
    for fut in done:
        try:
            batch = fut.result()
            if not isinstance(batch, Exception):
                docs.extend(batch)
        except Exception as exc:
            _rag_log.debug("RAG fetch task raised: %s", exc)

    _rag_log.info("RAG: %d chunks after fetch", len(docs))
    return docs


def _rag_fetch_all(results: list[dict], cfg: RAGConfig) -> list[Document]:
    """
    Sync entry-point — compatible with both Jupyter and plain scripts.
    Same brace-safety fix for the expansion prompt
    """
    try:
        loop = asyncio.get_running_loop()       # raises RuntimeError if not running
        return loop.run_until_complete(_rag_fetch_all_async(results, cfg))   # Jupyter path
    except RuntimeError:
        return asyncio.run(_rag_fetch_all_async(results, cfg))               # script path




def _rag_dense_retrieve(query: str, docs: list[Document], top_k: int) -> list[Document]:
    """Bi-encoder retrieval using the module-level SentenceTransformer (on-device, no network)."""
    if not docs:
        return []
    texts = [d.page_content for d in docs]
    doc_embs = _RAG_EMBED_MODEL.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    q_emb = _RAG_EMBED_MODEL.encode(query, normalize_embeddings=True)
    scores = np.dot(doc_embs, q_emb)
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [docs[i] for i in top_idx]


def _rag_rrf_fuse(ranked_lists: list[list[Document]], cfg: RAGConfig) -> list[Document]:
    """Reciprocal Rank Fusion — parameter-free merging of multiple ranked lists."""
    scores: dict[str, float] = {}
    doc_map: dict[str, Document] = {}
    for ranked in ranked_lists:
        for rank, doc in enumerate(ranked):
            key = "{}::{}".format(
                doc.metadata.get("url", ""),
                doc.metadata.get("chunk_id", hashlib.md5(doc.page_content.encode()).hexdigest()[:8]),
            )
            scores[key] = scores.get(key, 0.0) + 1.0 / (cfg.rrf_k + rank)
            doc_map[key] = doc
    ranked_keys = sorted(scores, key=scores.__getitem__, reverse=True)
    fused = []
    for key in ranked_keys:
        doc = doc_map[key]
        doc.metadata["rrf_score"] = scores[key]
        fused.append(doc)
    _rag_log.info("RAG RRF: merged -> %d docs", len(fused))
    return fused


def _rag_hybrid_retrieve(query: str, docs: list[Document], cfg: RAGConfig) -> list[Document]:
    bm25 = BM25Retriever.from_documents(docs, k=cfg.bm25_top_k)
    bm25_results = bm25.invoke(query)
    dense_results = _rag_dense_retrieve(query, docs, cfg.dense_top_k)
    return _rag_rrf_fuse([bm25_results, dense_results], cfg)




def _rag_diversity_filter(docs: list[Document], max_per_url: int) -> list[Document]:
    seen: dict[str, int] = {}
    filtered = []
    for doc in docs:
        url = doc.metadata.get("url", "")
        if seen.get(url, 0) >= max_per_url:
            continue
        filtered.append(doc)
        seen[url] = seen.get(url, 0) + 1
    return filtered


def _rag_cross_encode_rerank(query: str, docs: list[Document], cfg: RAGConfig) -> list[Document]:
    if not docs:
        return docs
    pairs = [(query, d.page_content) for d in docs]
    try:
        scores = _RAG_CROSS_ENCODER.predict(pairs)
    except Exception as exc:
        _rag_log.warning("RAG cross-encoder failed: %s", exc)
        return docs[: cfg.final_top_k]
    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    return [doc for _, doc in ranked[: cfg.final_top_k]]




def _rag_format_evidence(docs: list[Document]) -> str:
    if not docs:
        return "No evidence found."
    blocks = []
    for i, doc in enumerate(docs, 1):
        blocks.append(
            f"[{i}] {doc.metadata.get('title', 'N/A')}\n"
            f"URL: {doc.metadata.get('url', 'N/A')}\n"
            f"{doc.page_content}"
        )
    return "\n\n".join(blocks)


def _rag_evidence_sources(docs: list[Any]) -> list[dict[str, str]]:
    sources = []
    for doc in docs:
        sources.append(
            {
                "title": str(doc.metadata.get("title", ""))[:160],
                "url": str(doc.metadata.get("url", ""))[:300],
                "preview": " ".join(str(doc.page_content).split())[:300],
            }
        )
    return sources




def _rag_retrieve(query: str, cfg: RAGConfig, llm: LocalLLM) -> tuple[str, list[Any], float]:
    _ensure_rag_runtime()
    t0 = time.perf_counter()

    queries = _expand_query_rag(query, cfg, llm)

    raw_results = _rag_search_all(queries, cfg)
    if not raw_results:
        elapsed = time.perf_counter() - t0
        return "No results found.", [], elapsed

    docs = _rag_fetch_all(raw_results, cfg)
    if not docs:
        elapsed = time.perf_counter() - t0
        return "No results found.", [], elapsed

    fused_docs = _rag_hybrid_retrieve(query, docs, cfg)

    diverse_docs = _rag_diversity_filter(fused_docs, cfg.diversity_max_per_url)

    top_docs = _rag_cross_encode_rerank(query, diverse_docs, cfg)

    elapsed = time.perf_counter() - t0
    _rag_log.info("RAG retrieve() done in %.2fs -> %d chunks", elapsed, len(top_docs))
    return _rag_format_evidence(top_docs), top_docs, elapsed




class RAGStrategy(BaseStrategy):
    """
    Retrieval-Augmented Generation strategy.

    Plugs into the existing LocalLLM interface (GemmaLLM / QwenLLM).
    The same LLM is used for optional query expansion and for the final answer.

    Usage
    -----
        strategy = RAGStrategy(llm=GemmaLLM())
        prediction = strategy.answer(question)

        cfg = RAGConfig(num_extra_queries=0, final_top_k=3)
        strategy = RAGStrategy(llm=QwenLLM(), retrieval_config=cfg)
    """

    name = "rag"
    _log = _rag_log

    def __init__(
        self,
        llm: LocalLLM,
        retrieval_config: RAGConfig | None = None,
        fallback_strategy: BaseStrategy | None = None,
        retriever: Callable[[str, RAGConfig, LocalLLM], tuple[str, list[Any], float]] | None = None,
    ):
        self.llm = llm
        self.cfg = retrieval_config or RAGConfig()
        self.fallback = fallback_strategy or HeuristicStrategy()
        self.retriever = retriever or _rag_retrieve

    def answer(self, question: Question) -> AnswerPrediction:
        try:
            retrieval_query = _retrieval_query(question)
            evidence, evidence_docs, retrieval_seconds = self.retriever(retrieval_query, self.cfg, self.llm)
            prompt = build_rag_prompt(question, evidence)
            raw_text = self.llm.generate(prompt, max_new_tokens=self.cfg.answer_max_new_tokens)
            prediction = parse_answer_prediction(raw_text, question, strategy_name=self.name)
            rule_option = _domain_rule_option(question)
            if rule_option is not None and prediction.option_id != rule_option.id:
                prediction = AnswerPrediction(
                    option_id=rule_option.id,
                    answer_text=rule_option.text,
                    confidence=1.0,
                    reasoning="A general domain rule corrected the model answer.",
                    metadata={"strategy": self.name, "fallback": False, "rule_guard": True},
                )
            prediction.metadata.update({
                "strategy": self.name,
                "model_name": getattr(self.llm, "model_name", "unknown"),
                "device": getattr(self.llm, "device_summary", "unknown"),
                "retrieval_query": retrieval_query,
                "num_evidence_chunks": len(evidence_docs),
                "retrieval_seconds": round(retrieval_seconds, 2),
                "evidence_preview": evidence[:1200],
                "evidence_sources": _rag_evidence_sources(evidence_docs),
            })
            return prediction

        except Exception as exc:
            self._log.warning("RAGStrategy error, using fallback: %s", exc)
            fallback_pred = self.fallback.answer(question)
            fallback_pred.metadata["fallback"] = True
            fallback_pred.metadata["fallback_reason"] = str(exc)
            return fallback_pred

"""Game runner, JSONL logs, and small result summaries."""

import json
import time
from concurrent.futures import ThreadPoolExecutor, TimeoutError
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable



def from_client_question(client_question: Any) -> Question:
    return Question(
        id=int(getattr(client_question, "id")),
        text=str(getattr(client_question, "text")),
        options=[
            AnswerOption(id=int(getattr(option, "id")), text=str(getattr(option, "text")))
            for option in getattr(client_question, "options")
        ],
        level=int(getattr(client_question, "level", 0) or 0),
        metadata={"source": "millionaire_client"},
    )


class SafeDelay:
    def __init__(self, seconds: float = 1.0):
        self.seconds = max(0.0, seconds)
        self._last_call: float | None = None

    def wait(self) -> None:
        now = time.monotonic()
        if self._last_call is not None:
            remaining = self.seconds - (now - self._last_call)
            if remaining > 0:
                time.sleep(remaining)
        self._last_call = time.monotonic()


class GameRunner:
    def __init__(
        self,
        client: Any,
        safe_delay_seconds: float = 1.0,
        answer_timeout_seconds: float = 25.0,
        logger: "RunLogger | None" = None,
    ):
        self.client = client
        self.safe_delay = SafeDelay(safe_delay_seconds)
        self.answer_timeout_seconds = answer_timeout_seconds
        self.logger = logger

    def play(self, competition_id: int, strategy: BaseStrategy) -> Any:
        self.safe_delay.wait()
        game = self.client.game.start(competition_id=competition_id)
        while game.in_progress:
            if game.current_question is None:
                break
            question = from_client_question(game.current_question)
            started_at = time.monotonic()
            prediction = self._safe_answer(strategy, question)
            elapsed = time.monotonic() - started_at
            self.safe_delay.wait()
            try:
                result = game.answer(prediction.option_id)
            except Exception as exc:
                result = SubmissionErrorResult(error=str(exc))
                self._log(question, prediction, result, elapsed, strategy)
                break
            self._log(question, prediction, result, elapsed, strategy)
            if getattr(result, "game_over", False):
                break
        return game

    def _safe_answer(
        self,
        strategy: BaseStrategy,
        question: Question,
        timeout_seconds: float | None = None,
    ) -> AnswerPrediction:
        fallback = _fallback_prediction(question, "Strategy failed or timed out.")
        timeout_seconds = self.answer_timeout_seconds if timeout_seconds is None else timeout_seconds
        executor = ThreadPoolExecutor(max_workers=1)
        try:
            future = executor.submit(strategy.answer, question)
            prediction = future.result(timeout=timeout_seconds)
        except TimeoutError:
            executor.shutdown(wait=False, cancel_futures=True)
            fallback.metadata["error"] = f"Timed out after {timeout_seconds}s"
            return fallback
        except Exception as exc:
            executor.shutdown(wait=False, cancel_futures=True)
            fallback.metadata["error"] = str(exc)
            return fallback
        executor.shutdown(wait=False)
        if prediction.option_id not in question.valid_option_ids():
            fallback.metadata["error"] = "Strategy returned invalid option_id"
            fallback.metadata["original_prediction"] = prediction.metadata
            return fallback
        return prediction

    def _log(
        self,
        question: Question,
        prediction: AnswerPrediction,
        result: Any,
        elapsed_seconds: float,
        strategy: BaseStrategy,
    ) -> None:
        if self.logger:
            self.logger.log_attempt(
                question=question,
                prediction=prediction,
                result=result,
                elapsed_seconds=elapsed_seconds,
                strategy_name=getattr(strategy, "name", strategy.__class__.__name__),
            )


class SpeechGameRunner(GameRunner):
    def __init__(
        self,
        client: Any,
        safe_delay_seconds: float = 1.0,
        answer_timeout_seconds: float = 25.0,
        logger: "RunLogger | None" = None,
        transcriber: Callable[[bytes], str] | None = None,
        audio_dir: str | Path | None = None,
        audio_fetch_delay_seconds: float = 0.2,
    ):
        super().__init__(client, safe_delay_seconds, answer_timeout_seconds, logger)
        self.transcriber = transcriber
        self.audio_dir = Path(audio_dir) if audio_dir is not None else None
        self.audio_fetch_delay_seconds = max(0.0, audio_fetch_delay_seconds)

    def play(self, competition_id: int, strategy: BaseStrategy) -> Any:
        self.safe_delay.wait()
        game = self.client.game.start(competition_id=competition_id, mode="speech")
        while game.in_progress:
            if game.current_question is None:
                break
            self.safe_delay.wait()
            fetch_started_at = time.monotonic()
            question_audio = game.fetch_audio_question()
            self._sleep_between_audio_requests()
            option_audios = []
            for _ in range(len(getattr(game.current_question, "options", [])) or 4):
                option_audios.append(game.fetch_audio_option_next())
                self._sleep_between_audio_requests()
            audio_fetch_seconds = time.monotonic() - fetch_started_at
            if hasattr(game, "refresh_state"):
                try:
                    game.refresh_state()
                except Exception:
                    pass

            started_at = time.monotonic()
            question = self._question_from_audio(game, question_audio, option_audios, audio_fetch_seconds)
            remaining_before_strategy = getattr(game, "time_remaining", None)
            question.metadata["time_remaining_before_strategy"] = remaining_before_strategy
            strategy_timeout = self.answer_timeout_seconds
            if remaining_before_strategy is not None:
                strategy_timeout = min(strategy_timeout, max(1.0, float(remaining_before_strategy) - 1.0))
            if remaining_before_strategy is not None and remaining_before_strategy <= 1.0:
                prediction = _fallback_prediction(question, "Not enough time after speech transcription.")
            else:
                prediction = self._safe_answer(strategy, question, timeout_seconds=strategy_timeout)
            elapsed = time.monotonic() - started_at
            try:
                result = game.answer(prediction.option_id)
            except Exception as exc:
                result = SubmissionErrorResult(error=str(exc))
                self._log(question, prediction, result, elapsed, strategy)
                break
            self._log(question, prediction, result, elapsed, strategy)
            if getattr(result, "game_over", False):
                break
        return game

    def _question_from_audio(
        self,
        game: Any,
        question_audio: bytes,
        option_audios: list[bytes],
        audio_fetch_seconds: float,
    ) -> Question:
        client_question = game.current_question
        option_ids = [
            int(getattr(option, "id", index))
            for index, option in enumerate(getattr(client_question, "options", []))
        ]
        if not option_ids:
            option_ids = list(range(len(option_audios)))

        q_text, q_error = self._safe_transcribe(question_audio)
        option_texts = []
        option_errors = []
        for index, audio in enumerate(option_audios):
            text, error = self._safe_transcribe(audio)
            option_texts.append(text or f"Option {chr(65 + index)}")
            option_errors.append(error)

        question_id = int(getattr(client_question, "id", getattr(game, "current_level", 0) or 0))
        level = int(getattr(client_question, "level", getattr(game, "current_level", 0) or 0) or 0)
        question_text = q_text or "Speech question transcript unavailable."
        options = [
            AnswerOption(id=option_ids[index], text=option_texts[index])
            for index in range(min(len(option_ids), len(option_texts)))
        ]
        if not options:
            options = [AnswerOption(id=0, text="Option A")]
        self._save_audio(question_audio, question_id, level, "question")
        for index, audio in enumerate(option_audios):
            self._save_audio(audio, question_id, level, f"option_{chr(65 + index)}")
        return Question(
            id=question_id,
            text=question_text,
            options=options,
            level=level,
            metadata={
                "source": "millionaire_client",
                "mode": "speech",
                "audio_fetch_seconds": audio_fetch_seconds,
                "time_remaining_after_audio_fetch": getattr(game, "time_remaining", None),
                "question_transcription_error": q_error,
                "option_transcription_errors": option_errors,
            },
        )

    def _safe_transcribe(self, audio_bytes: bytes) -> tuple[str, str | None]:
        try:
            transcriber = self.transcriber
            if transcriber is None:
                raise RuntimeError("Speech transcription is not included in this text submission notebook.")
            return str(transcriber(audio_bytes)).strip(), None
        except Exception as exc:
            return "", str(exc)

    def _save_audio(self, audio_bytes: bytes, question_id: int, level: int, label: str) -> None:
        if self.audio_dir is None:
            return
        self.audio_dir.mkdir(parents=True, exist_ok=True)
        path = self.audio_dir / f"level_{level}_question_{question_id}_{label}.wav"
        path.write_bytes(audio_bytes)

    def _sleep_between_audio_requests(self) -> None:
        if self.audio_fetch_delay_seconds > 0:
            time.sleep(self.audio_fetch_delay_seconds)


class RunLogger:
    def __init__(self, path: str | Path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def log_attempt(
        self,
        question: Question,
        prediction: AnswerPrediction,
        result: Any,
        elapsed_seconds: float,
        strategy_name: str,
    ) -> None:
        row = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "strategy_name": strategy_name,
            "question": asdict(question),
            "prediction": asdict(prediction),
            "elapsed_seconds": elapsed_seconds,
            "result": {
                "correct": getattr(result, "correct", None),
                "game_over": getattr(result, "game_over", None),
                "earned_amount": getattr(result, "earned_amount", None),
                "timed_out": getattr(result, "timed_out", None),
                "status": getattr(result, "status", None),
                "current_level": getattr(result, "current_level", None),
                "error": getattr(result, "error", None),
            },
        }
        with self.path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def load_jsonl(path: str | Path) -> list[dict[str, Any]]:
    with Path(path).open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def summarize_attempts(rows: list[dict[str, Any]]) -> dict[str, Any]:
    total = len(rows)
    correct = sum(1 for row in rows if row.get("result", {}).get("correct") is True)
    timed_out = sum(1 for row in rows if row.get("result", {}).get("timed_out") is True)
    elapsed_values = [row.get("elapsed_seconds", 0.0) for row in rows]
    return {
        "total": total,
        "correct": correct,
        "accuracy": correct / total if total else 0.0,
        "timed_out": timed_out,
        "avg_elapsed_seconds": sum(elapsed_values) / total if total else 0.0,
    }


def benchmark_strategy(strategy: BaseStrategy, cases: list[tuple[Question, int]]) -> dict[str, Any]:
    rows = []
    for question, gold_id in cases:
        started_at = time.monotonic()
        prediction = strategy.answer(question)
        elapsed = time.monotonic() - started_at
        votes = prediction.metadata.get("votes") or []
        vote_options = {vote.get("option_id") for vote in votes if isinstance(vote, dict)}
        rows.append(
            {
                "question_id": question.id,
                "prediction": prediction,
                "gold_id": gold_id,
                "correct": prediction.option_id == gold_id,
                "elapsed_seconds": elapsed,
                "fallback": bool(prediction.metadata.get("fallback")),
                "disagreement": len(vote_options) > 1,
            }
        )

    total = len(rows)
    elapsed_values = [row["elapsed_seconds"] for row in rows]
    return {
        "total": total,
        "correct": sum(1 for row in rows if row["correct"]),
        "accuracy": sum(1 for row in rows if row["correct"]) / total if total else 0.0,
        "fallbacks": sum(1 for row in rows if row["fallback"]),
        "avg_elapsed_seconds": sum(elapsed_values) / total if total else 0.0,
        "max_elapsed_seconds": max(elapsed_values) if elapsed_values else 0.0,
        "disagreements": sum(1 for row in rows if row["disagreement"]),
        "rows": rows,
    }


def _fallback_prediction(question: Question, reason: str) -> AnswerPrediction:
    option = question.first_option()
    return AnswerPrediction(
        option_id=option.id,
        answer_text=option.text,
        confidence=0.0,
        reasoning=reason,
        metadata={"fallback": True},
    )


@dataclass
class SubmissionErrorResult:
    correct: bool | None = None
    game_over: bool = True
    earned_amount: float | None = None
    timed_out: bool = False
    status: str = "submission_error"
    current_level: int | None = None
    error: str = ""


## Run Settings


In [5]:
RUN_API_CHECK = False
RUN_LIVE_GAME = True
RUN_OFFLINE_BENCHMARK = False
BLOCK_LIVE_ON_BENCHMARK_FAILURE = False
PRELOAD_SELECTED_STRATEGIES = True
CLEAR_MEMORY_AFTER_EACH_MODEL = True
PROMPT_FOR_CREDENTIALS = False

RUN_BEST_BY_CATEGORY = True

API_URL = "http://131.175.15.22:51111/"
COMPETITION_IDS = [0, 1, 2, 3, 4, 5]
SAFE_DELAY_SECONDS = 2.0
ANSWER_TIMEOUT_SECONDS = 25.0

CATEGORY_NAMES = {
    0: "entertainment",
    1: "history",
    2: "science",
    3: "math",
    4: "philosophy_psychology",
    5: "news",
}

ARCHITECTURES = {
    "gemma_e2b_two_agent_quant_council": {
        "label": "Gemma E2B 4-bit tools + RAG council + quantized judge",
        "short_name": "gemma_e2b_two_agent",
        "kind": "gemma_tool_council_quant",
        "model_id": "google/gemma-4-E2B-it",
    },
    "qwen_two_agent_quant_council": {
        "label": "Qwen 3.5 2B 4-bit tools + RAG council + quantized judge",
        "short_name": "qwen_two_agent",
        "kind": "qwen_tool_council_quant",
        "model_id": "Qwen/Qwen3.5-2B",
    },
    "mixed_gemma_qwen_routed_rag": {
        "label": "Gemma + Qwen 4-bit mixed routed RAG",
        "short_name": "mixed_gemma_qwen",
        "kind": "mixed_quantized_rag",
        "model_id": "google/gemma-4-E2B-it + Qwen/Qwen3.5-2B",
    },
    "gemma_e2b_routed_rag": {
        "label": "Gemma E2B 4-bit tools + routed RAG",
        "short_name": "gemma_e2b_routed_rag",
        "kind": "gemma_tool_rag_quant",
        "model_id": "google/gemma-4-E2B-it",
    },
}

BEST_BY_CATEGORY_KEYS = {
    0: "qwen_two_agent_quant_council",
    1: "mixed_gemma_qwen_routed_rag",
    2: "gemma_e2b_two_agent_quant_council",
    3: "qwen_two_agent_quant_council",
    4: "gemma_e2b_routed_rag",
    5: "gemma_e2b_two_agent_quant_council",
}

BEST_OBSERVED_RESULTS = {
    0: {
        "earned": 32000,
        "architecture": "qwen_two_agent_quant_council",
        "log": "qwen3.5_2b_two-agent_quantized_council_quantized_judge_competition_0_20260530_141516.jsonl",
    },
    1: {
        "earned": 1024000,
        "architecture": "mixed_gemma_qwen_routed_rag",
        "log": "gemma_qwen_4-bit_mixed_routed_rag_competition_1_20260531_131927.jsonl",
    },
    2: {
        "earned": 1024000,
        "architecture": "gemma_e2b_two_agent_quant_council",
        "log": "category_best_2_science_gemma_e2b_two_agent_competition_2_20260531_171906.jsonl",
    },
    3: {
        "earned": 500,
        "architecture": "qwen_two_agent_quant_council",
        "log": "qwen_3.5_2b_4-bit_tools_rag_council_competition_3_20260531_141400.jsonl",
    },
    4: {
        "earned": 1024000,
        "architecture": "gemma_e2b_routed_rag",
        "log": "category_best_4_philosophy_psychology_gemma_e2b_routed_rag_competition_4_20260531_172300.jsonl",
    },
    5: {
        "earned": 2000,
        "architecture": "gemma_e2b_two_agent_quant_council",
        "log": "gemma_e2b_two-agent_quantized_council_quantized_judge_competition_5_20260528_155003.jsonl",
    },
}

print("API check:", RUN_API_CHECK, "Live game:", RUN_LIVE_GAME, "Benchmark:", RUN_OFFLINE_BENCHMARK)
print("Run category best:", RUN_BEST_BY_CATEGORY)
print("Block on benchmark failure:", BLOCK_LIVE_ON_BENCHMARK_FAILURE)
print("Competition IDs:", COMPETITION_IDS)
print("Selected architecture by category:")
for competition_id in COMPETITION_IDS:
    key = BEST_BY_CATEGORY_KEYS[competition_id]
    print("-", competition_id, CATEGORY_NAMES[competition_id], "->", ARCHITECTURES[key]["label"])

print("Best observed result by category:")
for competition_id in COMPETITION_IDS:
    observed = BEST_OBSERVED_RESULTS[competition_id]
    print(
        "-",
        competition_id,
        CATEGORY_NAMES[competition_id],
        "->",
        observed["earned"],
        "with",
        ARCHITECTURES[observed["architecture"]]["label"],
    )


API check: False Live game: True Benchmark: False
Run category best: True
Block on benchmark failure: False
Competition IDs: [0, 1, 2, 3, 4, 5]
Selected architecture by category:
- 0 entertainment -> Qwen 3.5 2B 4-bit tools + RAG council + quantized judge
- 1 history -> Gemma + Qwen 4-bit mixed routed RAG
- 2 science -> Gemma E2B 4-bit tools + RAG council + quantized judge
- 3 math -> Qwen 3.5 2B 4-bit tools + RAG council + quantized judge
- 4 philosophy_psychology -> Gemma E2B 4-bit tools + routed RAG
- 5 news -> Gemma E2B 4-bit tools + RAG council + quantized judge
Best observed result by category:
- 0 entertainment -> 32000 with Qwen 3.5 2B 4-bit tools + RAG council + quantized judge
- 1 history -> 1024000 with Gemma + Qwen 4-bit mixed routed RAG
- 2 science -> 1024000 with Gemma E2B 4-bit tools + RAG council + quantized judge
- 3 math -> 500 with Qwen 3.5 2B 4-bit tools + RAG council + quantized judge
- 4 philosophy_psychology -> 1024000 with Gemma E2B 4-bit tools + routed RAG
- 5 

## Login


In [6]:
import getpass


try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

USERNAME = os.environ.get("POLIMILLIONAIRE_USERNAME")
PASSWORD = os.environ.get("POLIMILLIONAIRE_PASSWORD")
try:
    from google.colab import userdata
    USERNAME = USERNAME or userdata.get("POLIMILLIONAIRE_USERNAME")
    PASSWORD = PASSWORD or userdata.get("POLIMILLIONAIRE_PASSWORD")
except Exception:
    pass
if PROMPT_FOR_CREDENTIALS and not USERNAME:
    USERNAME = input("PoliMillionaire username: ").strip()
if PROMPT_FOR_CREDENTIALS and not PASSWORD:
    PASSWORD = getpass.getpass("PoliMillionaire password: ")
print("Username configured:", bool(USERNAME))
print("Password configured:", bool(PASSWORD))


Username configured: True
Password configured: True


## API Check


In [7]:
client = None
if RUN_API_CHECK:
    if not USERNAME or not PASSWORD:
        raise RuntimeError("Set credentials first.")
    try:
        client = MillionaireClient(API_URL)
        user = client.login(USERNAME, PASSWORD)
        print("Logged in as", user.username)
        for competition in client.competitions.list_all():
            print(competition.id, competition.name, competition.max_levels)
    except AuthenticationError as exc:
        client = None
        print("Login failed:", exc)
else:
    print("API check skipped.")


API check skipped.


## Benchmark Questions


In [8]:
import re


warmup_question = Question(1, "What is 2 + 2?", [AnswerOption(0, "3"), AnswerOption(1, "4"), AnswerOption(2, "5"), AnswerOption(3, "22")])
rag_warmup_question = Question(4, "In which year was A Haunted House 2 released?", [AnswerOption(0, "2012"), AnswerOption(1, "2015"), AnswerOption(2, "2014"), AnswerOption(3, "2013")])
TUTORIAL_SET = [
    (warmup_question, 1),
    (rag_warmup_question, 2),
    (Question(2, "Which song was NOT written by Bob Dylan?", [AnswerOption(0, "Like a Rolling Stone"), AnswerOption(1, "Blowin' in the Wind"), AnswerOption(2, "The Times They Are A-Changin'"), AnswerOption(3, "Hound Dog")]), 3),
    (Question(3, "What was Whitney Houston's debut album?", [AnswerOption(0, "Whitney Houston"), AnswerOption(1, "Just Whitney"), AnswerOption(2, "I'm Your Baby Tonight"), AnswerOption(3, "Whitney")]), 0),
    (Question(5, "Compute \\dbinom{85}{82}.", [AnswerOption(0, "252"), AnswerOption(1, "101170"), AnswerOption(2, "98770"), AnswerOption(3, "4680")]), 2),
    (Question(6, "Which genetics principle says alleles for one trait separate independently from alleles for another trait?", [AnswerOption(0, "Law of independent assortment"), AnswerOption(1, "Law of dominance"), AnswerOption(2, "Genetic drift"), AnswerOption(3, "Mitochondrial inheritance")]), 0),
    (Question(7, "Which of the following is a key characteristic of hard bop?", [AnswerOption(0, "Incorporation of blues and gospel elements"), AnswerOption(1, "Use of electronic synthesizers"), AnswerOption(2, "No structured form"), AnswerOption(3, "Strict adherence to 4/4 time")]), 0),
    (Question(8, "For a Japan-set racing game, which research step best supports authentic representation?", [AnswerOption(0, "Only copying older racing games"), AnswerOption(1, "Working with local cultural consultants and doing field research in Japan"), AnswerOption(2, "Using random online comments"), AnswerOption(3, "Avoiding real Japanese locations")]), 1),
    (Question(9, "In bystander-effect research, how can strong group cohesiveness change helping behavior?", [AnswerOption(0, "It can make people more likely to help members of the group"), AnswerOption(1, "It always removes personal responsibility"), AnswerOption(2, "It makes intervention impossible"), AnswerOption(3, "It has no relationship to helping")]), 0),
    (Question(10, "What is the usual link between the Battle of Marathon and the Older Parthenon?", [AnswerOption(0, "The temple project is linked to Athens after its victory at Marathon"), AnswerOption(1, "The battle was fought inside the Parthenon"), AnswerOption(2, "The Parthenon was built by Sparta after Marathon"), AnswerOption(3, "The Older Parthenon was a Roman project")]), 0),
    (Question(11, "Why was Hitchcock involved with the British Ministry of Information documentary about liberated concentration camps?", [AnswerOption(0, "To make a comedy film"), AnswerOption(1, "To help present visual evidence of Nazi atrocities clearly"), AnswerOption(2, "To advertise a commercial movie"), AnswerOption(3, "To hide the camps from the public")]), 1),
]
CATEGORY_BENCHMARKS = {
    0: [
        TUTORIAL_SET[2],
        TUTORIAL_SET[3],
        TUTORIAL_SET[6],
        TUTORIAL_SET[10],
    ],
    1: [
        TUTORIAL_SET[9],
        (Question(15, "Which event is usually associated with the end of the Roman Republic and the rise of the Roman Empire?", [AnswerOption(0, "The accession of Augustus as first emperor"), AnswerOption(1, "The founding of Rome"), AnswerOption(2, "The fall of Constantinople"), AnswerOption(3, "The Punic Wars")]), 0),
    ],
    2: [
        TUTORIAL_SET[5],
        (Question(12, "An organ pipe produces a note with wavelength 2.67 m. If the speed of sound is 343 m/s, what is the frequency?", [AnswerOption(0, "85.7 Hz"), AnswerOption(1, "128 Hz"), AnswerOption(2, "256 Hz"), AnswerOption(3, "343 Hz")]), 1),
        (Question(16, "Materials such as carbon and nitrogen move repeatedly through living things and the environment. What is this process called?", [AnswerOption(0, "A biogeochemical cycle"), AnswerOption(1, "A chemical explosion"), AnswerOption(2, "A nuclear chain reaction"), AnswerOption(3, "A food label")]), 0),
    ],
    3: [
        TUTORIAL_SET[4],
        (Question(13, "How many homomorphisms are there of Z into Z_2?", [AnswerOption(0, "infinitely many"), AnswerOption(1, "0"), AnswerOption(2, "1"), AnswerOption(3, "2")]), 3),
        (Question(14, "A producer compares acne scores on the same people before and after a new cream. Which test is appropriate?", [AnswerOption(0, "A two-sample t-test"), AnswerOption(1, "A matched pairs t-test"), AnswerOption(2, "A chi-square test"), AnswerOption(3, "A two-proportion z-test")]), 1),
        (Question(17, "What is 1^2 + 2^2 + 3^2 + 4^2?", [AnswerOption(0, "16"), AnswerOption(1, "20"), AnswerOption(2, "30"), AnswerOption(3, "40")]), 2),
    ],
    4: [
        TUTORIAL_SET[8],
        (Question(18, "What is the main difference between realism and anti-realism in philosophy?", [AnswerOption(0, "Realism says reality exists independently of us; anti-realism ties truth to minds or theories"), AnswerOption(1, "Both deny any external world"), AnswerOption(2, "Realism is only about painting"), AnswerOption(3, "Anti-realism says science is always impossible")]), 0),
        (Question(19, "Which idea is central to Stoic ethics?", [AnswerOption(0, "Virtue is the only true good"), AnswerOption(1, "Pleasure is the only good"), AnswerOption(2, "Wealth is the only good"), AnswerOption(3, "Fame is the only good")]), 0),
    ],
    5: [
        (Question(20, "According to this report summary, a broadcast deal stalled because regulators had not approved the rights package. What reason does the summary give?", [AnswerOption(0, "Regulators had not approved the rights package"), AnswerOption(1, "The sport was cancelled"), AnswerOption(2, "The teams refused to play"), AnswerOption(3, "There were too many stadiums")]), 0),
        (Question(21, "According to this news summary, a housing shortage mainly affected lower and middle price ranges. Which range was affected?", [AnswerOption(0, "Only luxury apartments"), AnswerOption(1, "Lower and middle price ranges"), AnswerOption(2, "Only vacation homes"), AnswerOption(3, "No price range")]), 1),
    ],
}
CATEGORY_PROBES = CATEGORY_BENCHMARKS
BENCHMARK_SET = [(category, question, gold_id) for category, items in CATEGORY_BENCHMARKS.items() for question, gold_id in items]


## Model Cleanup


In [9]:
def clear_model_memory(strategy=None, release_rag=False):
    try:
        unload_strategy(strategy)
        if release_rag:
            unload_rag_runtime()
        gc.collect()
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception as exc:
        print("Cleanup warning:", exc)


## Architecture Builders


In [10]:
def mixed_council(quantized=False):
    if quantized and importlib.util.find_spec("bitsandbytes") is None:
        raise RuntimeError("For the 4-bit council, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
    gemma = GemmaLLM(GemmaLLMConfig(model_id="google/gemma-4-E2B-it", quantize_4bit=quantized, max_new_tokens=32, do_sample=True, seed=42, generation_max_time_seconds=6.0, timeout_seconds=120.0))
    qwen = QwenLLM(QwenLLMConfig(model_id="Qwen/Qwen3.5-2B", quantize_4bit=quantized, max_new_tokens=32, temperature=0.7, top_p=0.8, top_k=20, do_sample=True, seed=43, enable_thinking=False, generation_max_time_seconds=6.0, timeout_seconds=120.0))
    return CouncilStrategy(candidate_llms=[gemma, qwen], judge_llm=gemma, judge_scope="candidate_only" if quantized else "any_option", rejected_judge_fallback="primary_candidate" if quantized else "confidence_weighted", base_seed=200, candidate_max_new_tokens=32, judge_max_new_tokens=8, max_time_per_call=6.0)


def mixed_quantized_routed_rag():
    if importlib.util.find_spec("bitsandbytes") is None:
        raise RuntimeError("For 4-bit mixed routed RAG, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
    gemma = GemmaLLM(GemmaLLMConfig(model_id="google/gemma-4-E2B-it", quantize_4bit=True, max_new_tokens=48, temperature=0.0, do_sample=False, seed=42, generation_max_time_seconds=12.0, timeout_seconds=120.0))
    qwen = QwenLLM(QwenLLMConfig(model_id="Qwen/Qwen3.5-2B", quantize_4bit=True, max_new_tokens=48, temperature=0.7, top_p=0.8, top_k=20, do_sample=True, seed=43, enable_thinking=False, generation_max_time_seconds=10.0, timeout_seconds=120.0))
    gemma_direct = LocalLLMVariant(gemma, "gemma-e2b-direct", do_sample=False, temperature=0.0, seed=42, max_time=8.0)
    gemma_candidate = LocalLLMVariant(gemma, "gemma-e2b-candidate", do_sample=True, temperature=0.55, top_p=0.9, seed=201, max_time=10.0)
    direct = CalculatorStrategy(GemmaStrategy(llm=gemma_direct))
    council = CouncilStrategy(candidate_llms=[gemma_candidate, qwen], judge_llm=gemma_direct, judge_scope="candidate_only", rejected_judge_fallback="primary_candidate", base_seed=200, candidate_max_new_tokens=48, judge_max_new_tokens=8, max_time_per_call=10.0)
    rag = RAGStrategy(llm=gemma_direct, retrieval_config=RAGConfig(num_extra_queries=0, answer_max_new_tokens=80), fallback_strategy=council)
    return RoutedStrategy(direct_strategy=direct, rag_strategy=rag, fallback_strategy=council, low_confidence_strategy=council, rag_min_confidence=0.65)


def gemma_e2b_rag_council_e4b_judge():
    if importlib.util.find_spec("bitsandbytes") is None:
        raise RuntimeError("For the Gemma council, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
    e2b = GemmaLLM(GemmaLLMConfig(model_id="google/gemma-4-E2B-it", quantize_4bit=True, max_new_tokens=80, temperature=0.7, do_sample=True, generation_max_time_seconds=12.0, timeout_seconds=120.0))
    e4b_judge = GemmaLLM(GemmaLLMConfig(model_id="google/gemma-4-E4B-it", quantize_4bit=True, max_new_tokens=16, temperature=0.0, do_sample=False, seed=900, generation_max_time_seconds=10.0, timeout_seconds=120.0))
    candidates = [
        LocalLLMVariant(e2b, "gemma-e2b-agent-a", do_sample=True, temperature=0.35, top_p=0.9, seed=501, max_time=10.0),
        LocalLLMVariant(e2b, "gemma-e2b-agent-b", do_sample=True, temperature=0.65, top_p=0.9, seed=502, max_time=10.0),
        LocalLLMVariant(e2b, "gemma-e2b-agent-c", do_sample=True, temperature=0.9, top_p=0.95, seed=503, max_time=10.0),
    ]
    direct = CalculatorStrategy(GemmaStrategy(llm=LocalLLMVariant(e2b, "gemma-e2b-direct", do_sample=False, temperature=0.0, seed=42, max_time=8.0)))
    return RoutedRAGCouncilStrategy(
        candidate_llms=candidates,
        judge_llm=e4b_judge,
        direct_strategy=direct,
        retrieval_config=RAGConfig(num_extra_queries=0, max_ddg_results=3, timeout_ddg=4, fetch_timeout=3.0, total_fetch_budget=3.0, bm25_top_k=4, dense_top_k=4, final_top_k=2, answer_max_new_tokens=48),
        judge_scope="candidate_only",
        base_seed=600,
        candidate_max_new_tokens=48,
        judge_max_new_tokens=8,
        max_time_per_call=8.0,
        always_judge=False,
        unload_candidates_before_judge=True,
    )



def gemma_e2b_quantized_council_e2b_normal_judge():
    if importlib.util.find_spec("bitsandbytes") is None:
        raise RuntimeError("For this council, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
    e2b = GemmaLLM(GemmaLLMConfig(model_id="google/gemma-4-E2B-it", quantize_4bit=True, max_new_tokens=48, temperature=0.7, do_sample=True, generation_max_time_seconds=8.0, timeout_seconds=120.0))
    candidates = [
        LocalLLMVariant(e2b, "gemma-e2b-4bit-evidence-checker", do_sample=True, temperature=0.35, top_p=0.9, seed=701, max_time=8.0),
        LocalLLMVariant(e2b, "gemma-e2b-4bit-option-eliminator", do_sample=True, temperature=0.75, top_p=0.95, seed=702, max_time=8.0),
    ]
    judge = LocalLLMVariant(e2b, "gemma-e2b-4bit-judge", do_sample=False, temperature=0.0, seed=900, max_time=8.0)
    direct = CalculatorStrategy(GemmaStrategy(llm=LocalLLMVariant(e2b, "gemma-e2b-4bit-direct", do_sample=False, temperature=0.0, seed=42, max_time=8.0)))
    return RoutedRAGCouncilStrategy(
        candidate_llms=candidates,
        judge_llm=judge,
        direct_strategy=direct,
        retrieval_config=RAGConfig(num_extra_queries=0, max_ddg_results=3, timeout_ddg=4, fetch_timeout=3.0, total_fetch_budget=3.0, bm25_top_k=4, dense_top_k=4, final_top_k=2, answer_max_new_tokens=48),
        judge_scope="candidate_only",
        base_seed=700,
        candidate_styles=["evidence_checker", "option_eliminator"],
        candidate_max_new_tokens=48,
        judge_max_new_tokens=8,
        max_time_per_call=8.0,
        always_judge=False,
        unload_candidates_before_judge=False,
        unload_judge_before_candidates=False,
    )


def qwen_quantized_rag_council():
    if importlib.util.find_spec("bitsandbytes") is None:
        raise RuntimeError("For this Qwen council, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
    qwen = QwenLLM(QwenLLMConfig(model_id="Qwen/Qwen3.5-2B", quantize_4bit=True, max_new_tokens=48, temperature=0.7, top_p=0.9, top_k=20, do_sample=True, seed=42, enable_thinking=False, generation_max_time_seconds=8.0, timeout_seconds=120.0))
    candidates = [
        LocalLLMVariant(qwen, "qwen-3.5-2b-4bit-evidence-checker", do_sample=True, temperature=0.25, top_p=0.9, top_k=20, seed=801, max_time=8.0),
        LocalLLMVariant(qwen, "qwen-3.5-2b-4bit-option-eliminator", do_sample=True, temperature=0.75, top_p=0.95, top_k=20, seed=802, max_time=8.0),
    ]
    judge = LocalLLMVariant(qwen, "qwen-3.5-2b-4bit-judge", do_sample=False, seed=950, max_time=8.0)
    direct_llm = LocalLLMVariant(qwen, "qwen-3.5-2b-4bit-direct", do_sample=False, seed=42, max_time=8.0)
    direct = CalculatorStrategy(QwenStrategy(llm=direct_llm, model_config=QwenLLMConfig(enable_thinking=False)))
    return RoutedRAGCouncilStrategy(
        candidate_llms=candidates,
        judge_llm=judge,
        direct_strategy=direct,
        retrieval_config=RAGConfig(num_extra_queries=0, max_ddg_results=3, timeout_ddg=4, fetch_timeout=3.0, total_fetch_budget=3.0, bm25_top_k=4, dense_top_k=4, final_top_k=2, answer_max_new_tokens=48),
        judge_scope="candidate_only",
        base_seed=800,
        candidate_styles=["evidence_checker", "option_eliminator"],
        candidate_max_new_tokens=48,
        judge_max_new_tokens=8,
        max_time_per_call=8.0,
        always_judge=False,
        unload_candidates_before_judge=False,
        unload_judge_before_candidates=False,
    )

def gemma_quantized_routed_rag(model_id="google/gemma-4-E2B-it", label="gemma-e2b"):
    if importlib.util.find_spec("bitsandbytes") is None:
        raise RuntimeError("For Gemma routed RAG, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
    gemma = GemmaLLM(GemmaLLMConfig(model_id=model_id, quantize_4bit=True, max_new_tokens=48, temperature=0.0, do_sample=False, seed=42, generation_max_time_seconds=10.0, timeout_seconds=120.0))
    direct_llm = LocalLLMVariant(gemma, f"{label}-rag-direct", do_sample=False, temperature=0.0, seed=42, max_time=8.0)
    direct = CalculatorStrategy(GemmaStrategy(llm=direct_llm))
    rag = RAGStrategy(
        llm=direct_llm,
        retrieval_config=RAGConfig(num_extra_queries=0, max_ddg_results=3, timeout_ddg=4, fetch_timeout=3.0, total_fetch_budget=3.0, bm25_top_k=4, dense_top_k=4, final_top_k=2, answer_max_new_tokens=64),
        fallback_strategy=direct,
    )
    return RoutedStrategy(direct_strategy=direct, rag_strategy=rag, fallback_strategy=direct, low_confidence_strategy=direct, rag_min_confidence=0.55)


def qwen_quantized_tool_rag():
    if importlib.util.find_spec("bitsandbytes") is None:
        raise RuntimeError("For Qwen routed RAG, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
    qwen = QwenLLM(QwenLLMConfig(model_id="Qwen/Qwen3.5-2B", quantize_4bit=True, max_new_tokens=48, temperature=0.0, do_sample=False, seed=42, enable_thinking=False, generation_max_time_seconds=10.0, timeout_seconds=120.0))
    direct_llm = LocalLLMVariant(qwen, "qwen-3.5-2b-rag-direct", do_sample=False, seed=42, max_time=8.0)
    direct = CalculatorStrategy(QwenStrategy(llm=direct_llm, model_config=QwenLLMConfig(enable_thinking=False)))
    rag = RAGStrategy(
        llm=direct_llm,
        retrieval_config=RAGConfig(num_extra_queries=0, max_ddg_results=3, timeout_ddg=4, fetch_timeout=3.0, total_fetch_budget=3.0, bm25_top_k=4, dense_top_k=4, final_top_k=2, answer_max_new_tokens=64),
        fallback_strategy=direct,
    )
    return RoutedStrategy(direct_strategy=direct, rag_strategy=rag, fallback_strategy=direct, low_confidence_strategy=direct, rag_min_confidence=0.55)


class LazyFrankenFallback:
    name = "lazy_franken_fallback"

    def __init__(self, owner, key):
        self.owner = owner
        self.key = key

    def answer(self, question):
        return self.owner._get(self.key).answer(question)


class DataRoutedFrankenStrategy:
    name = "data_routed_franken"

    def __init__(self):
        self._loaded = {}
        self._models = {}

    def loaded_strategies(self):
        return list(self._loaded.values())

    def unload(self):
        for strategy in list(self._loaded.values()):
            unload_strategy(strategy)
        for model in list(self._models.values()):
            unload_strategy(model)
        unload_rag_runtime()
        self._loaded.clear()
        self._models.clear()

    def _gemma_llm(self):
        if "gemma" not in self._models:
            self._models["gemma"] = GemmaLLM(GemmaLLMConfig(
                model_id="google/gemma-4-E2B-it",
                quantize_4bit=True,
                max_new_tokens=48,
                temperature=0.0,
                do_sample=False,
                seed=42,
                generation_max_time_seconds=10.0,
                timeout_seconds=120.0,
            ))
        return self._models["gemma"]

    def _qwen_llm(self):
        if "qwen" not in self._models:
            self._models["qwen"] = QwenLLM(QwenLLMConfig(
                model_id="Qwen/Qwen3.5-2B",
                quantize_4bit=True,
                max_new_tokens=48,
                temperature=0.7,
                top_p=0.9,
                top_k=20,
                do_sample=True,
                seed=42,
                enable_thinking=False,
                generation_max_time_seconds=8.0,
                timeout_seconds=120.0,
            ))
        return self._models["qwen"]

    def _build_gemma_rag(self):
        gemma = self._gemma_llm()
        direct_llm = LocalLLMVariant(gemma, "franken-gemma-rag-direct", do_sample=False, temperature=0.0, seed=42, max_time=8.0)
        direct = CalculatorStrategy(GemmaStrategy(llm=direct_llm))
        rag = RAGStrategy(
            llm=direct_llm,
            retrieval_config=RAGConfig(num_extra_queries=0, max_ddg_results=3, timeout_ddg=4, fetch_timeout=3.0, total_fetch_budget=3.0, bm25_top_k=4, dense_top_k=4, final_top_k=2, answer_max_new_tokens=64),
            fallback_strategy=direct,
        )
        return RoutedStrategy(direct_strategy=direct, rag_strategy=rag, fallback_strategy=direct, low_confidence_strategy=direct, rag_min_confidence=0.55)

    def _build_gemma_council(self):
        gemma = self._gemma_llm()
        candidates = [
            LocalLLMVariant(gemma, "franken-gemma-evidence-checker", do_sample=True, temperature=0.35, top_p=0.9, seed=701, max_time=8.0),
            LocalLLMVariant(gemma, "franken-gemma-option-eliminator", do_sample=True, temperature=0.75, top_p=0.95, seed=702, max_time=8.0),
        ]
        judge = LocalLLMVariant(gemma, "franken-gemma-judge", do_sample=False, temperature=0.0, seed=900, max_time=8.0)
        direct = CalculatorStrategy(GemmaStrategy(llm=LocalLLMVariant(gemma, "franken-gemma-direct", do_sample=False, temperature=0.0, seed=42, max_time=8.0)))
        return RoutedRAGCouncilStrategy(
            candidate_llms=candidates,
            judge_llm=judge,
            direct_strategy=direct,
            retrieval_config=RAGConfig(num_extra_queries=0, max_ddg_results=3, timeout_ddg=4, fetch_timeout=3.0, total_fetch_budget=3.0, bm25_top_k=4, dense_top_k=4, final_top_k=2, answer_max_new_tokens=48),
            judge_scope="candidate_only",
            base_seed=700,
            candidate_styles=["evidence_checker", "option_eliminator"],
            candidate_max_new_tokens=48,
            judge_max_new_tokens=8,
            max_time_per_call=8.0,
            always_judge=False,
            unload_candidates_before_judge=False,
            unload_judge_before_candidates=False,
        )

    def _build_qwen_council(self):
        qwen = self._qwen_llm()
        candidates = [
            LocalLLMVariant(qwen, "franken-qwen-evidence-checker", do_sample=True, temperature=0.25, top_p=0.9, top_k=20, seed=801, max_time=8.0),
            LocalLLMVariant(qwen, "franken-qwen-option-eliminator", do_sample=True, temperature=0.75, top_p=0.95, top_k=20, seed=802, max_time=8.0),
        ]
        judge = LocalLLMVariant(qwen, "franken-qwen-judge", do_sample=False, seed=950, max_time=8.0)
        direct_llm = LocalLLMVariant(qwen, "franken-qwen-direct", do_sample=False, seed=42, max_time=8.0)
        direct = CalculatorStrategy(QwenStrategy(llm=direct_llm, model_config=QwenLLMConfig(enable_thinking=False)))
        return RoutedRAGCouncilStrategy(
            candidate_llms=candidates,
            judge_llm=judge,
            direct_strategy=direct,
            retrieval_config=RAGConfig(num_extra_queries=0, max_ddg_results=3, timeout_ddg=4, fetch_timeout=3.0, total_fetch_budget=3.0, bm25_top_k=4, dense_top_k=4, final_top_k=2, answer_max_new_tokens=48),
            judge_scope="candidate_only",
            base_seed=800,
            candidate_styles=["evidence_checker", "option_eliminator"],
            candidate_max_new_tokens=48,
            judge_max_new_tokens=8,
            max_time_per_call=8.0,
            always_judge=False,
            unload_candidates_before_judge=False,
            unload_judge_before_candidates=False,
        )

    def _get(self, key):
        if key not in self._loaded:
            if key == "gemma_council":
                self._loaded[key] = self._build_gemma_council()
            elif key == "qwen_council":
                self._loaded[key] = self._build_qwen_council()
            elif key == "gemma_rag":
                self._loaded[key] = self._build_gemma_rag()
            elif key == "tools":
                self._loaded[key] = CalculatorStrategy(fallback_strategy=LazyFrankenFallback(self, "gemma_council"))
            else:
                raise KeyError(key)
        return self._loaded[key]

    def _route(self, question):
        text = " ".join([question.text, *(option.text for option in question.options)]).lower()
        decision = route_question(question)
        if decision.reason == "math" or any(term in text for term in ["frequency", "wavelength", "homomorphism", "significance test", "matched pairs", "binom", "dbinom", "linear transformation", "confidence interval", "interquartile range", "decreased linearly"]):
            return "tools", "tool_math_or_formula"
        if re.search(r"\b20\d{2}[-/]\d{1,2}[-/]\d{1,2}\b", text) or any(term in text for term in ["according to the report", "news article", "broadcasting deal", "reported that", "coach stated"]):
            return "gemma_rag", "news_or_report_gemma_rag"
        if any(term in text for term in ["philosophy", "psychology", "bystander", "realism", "anti-realism", "denial", "santayana", "aristotle", "kant", "nietzsche", "plato", "blindsight", "relationship satisfaction"]):
            return "qwen_council", "philosophy_or_psych_qwen"
        if any(term in text for term in ["allele", "genetic", "biology", "cloud", "precipitation", "organ pipe", "wavelength", "molecule", "embryo", "frequency", "speed of sound", "planet", "solar system", "reef", "ecosystem"]):
            return "gemma_council", "science_gemma_council"
        if any(term in text for term in ["film", "movie", "song", "album", "music", "actor", "actress", "director", "singer", "dylan", "hitchcock", "kubrick", "cameron"]):
            return "gemma_rag", "entertainment_gemma_rag"
        if any(term in text for term in ["roman", "ancient", "alexander", "egyptian", "empire", "historical", "history"]):
            return "gemma_council", "history_gemma_council"
        return "gemma_council", "default_gemma"

    def answer(self, question):
        key, reason = self._route(question)
        prediction = self._get(key).answer(question)
        prediction.metadata["franken_component"] = key
        prediction.metadata["franken_route"] = reason
        return prediction


    def preload_components(self):
        probes = [
            ("tools", warmup_question, 1),
            ("gemma_rag", Question(6, "Which genetics principle says alleles for one trait separate independently from alleles for another trait?", [AnswerOption(0, "Law of independent assortment"), AnswerOption(1, "Law of dominance"), AnswerOption(2, "Genetic drift"), AnswerOption(3, "Mitochondrial inheritance")]), 0),
            ("qwen_council", Question(9, "In bystander-effect research, how can strong group cohesiveness change helping behavior?", [AnswerOption(0, "It can make people more likely to help members of the group"), AnswerOption(1, "It always removes personal responsibility"), AnswerOption(2, "It makes intervention impossible"), AnswerOption(3, "It has no relationship to helping")]), 0),
            ("gemma_council", Question(11, "Why was Hitchcock involved with the British Ministry of Information documentary about liberated concentration camps?", [AnswerOption(0, "To make a comedy film"), AnswerOption(1, "To help present visual evidence of Nazi atrocities clearly"), AnswerOption(2, "To advertise a commercial movie"), AnswerOption(3, "To hide the camps from the public")]), 1),
        ]
        rows = []
        for component, question, gold_id in probes:
            started = time.monotonic()
            prediction = self.answer(question)
            seconds = time.monotonic() - started
            rows.append({
                "component": component,
                "seconds": round(seconds, 2),
                "correct": prediction.option_id == gold_id,
                "fallback": bool(prediction.metadata.get("fallback")),
                "predicted": prediction.option_id,
                "gold": gold_id,
            })
        return rows


## Strategy Factory


In [11]:
def has_rag_strategy(strategy):
    if isinstance(strategy, DataRoutedFrankenStrategy):
        return True
    if isinstance(strategy, (RAGStrategy, RoutedRAGCouncilStrategy)):
        return True
    if isinstance(strategy, RoutedStrategy):
        return has_rag_strategy(strategy.rag_strategy)
    return False


def strategy_devices(strategy):
    devices = []

    def collect(item):
        if item is None:
            return
        if isinstance(item, DataRoutedFrankenStrategy):
            for loaded in item.loaded_strategies():
                collect(loaded)
            return
        if isinstance(item, CouncilStrategy):
            for llm in item.candidate_llms + [item.judge_llm]:
                devices.append(getattr(llm, "device_summary", "unknown"))
            return
        if isinstance(item, RoutedRAGCouncilStrategy):
            collect(item.direct_strategy)
            for llm in item.candidate_llms + [item.judge_llm]:
                devices.append(getattr(llm, "device_summary", "unknown"))
            return
        if isinstance(item, RoutedStrategy):
            collect(item.direct_strategy)
            collect(item.rag_strategy)
            collect(item.fallback_strategy)
            collect(item.low_confidence_strategy)
            return
        if hasattr(item, "llm"):
            devices.append(getattr(item.llm, "device_summary", "unknown"))

    collect(strategy)
    return list(dict.fromkeys(devices))


QUANTIZED_KINDS = {"gemma_quantized", "mixed_quantized", "mixed_quantized_rag", "gemma_e2b_rag_council_e4b_judge", "gemma_e2b_quantized_council_e2b_normal_judge", "qwen_quantized_rag_council", "franken_data_route", "rag", "gemma_tool_rag_quant", "gemma_e4b_tool_rag_quant", "qwen_tool_rag_quant", "gemma_tool_council_quant", "qwen_tool_council_quant"}


def make_strategy(item):
    if item["kind"] == "tool_heuristic":
        fallback = HeuristicStrategy()
        return RoutedStrategy(direct_strategy=CalculatorStrategy(fallback), fallback_strategy=fallback)
    if item["kind"] == "gemma_tool_rag_quant":
        return gemma_quantized_routed_rag(model_id="google/gemma-4-E2B-it", label="gemma-e2b")
    if item["kind"] == "gemma_e4b_tool_rag_quant":
        return gemma_quantized_routed_rag(model_id="google/gemma-4-E4B-it", label="gemma-e4b")
    if item["kind"] == "qwen_tool_rag_quant":
        return qwen_quantized_tool_rag()
    if item["kind"] == "gemma_tool_council_quant":
        return gemma_e2b_quantized_council_e2b_normal_judge()
    if item["kind"] == "qwen_tool_council_quant":
        return qwen_quantized_rag_council()
    if item["kind"] == "franken_data_route":
        return DataRoutedFrankenStrategy()
    if item["kind"] == "qwen_quantized_rag_council":
        return qwen_quantized_rag_council()
    if item["kind"] == "gemma_e2b_quantized_council_e2b_normal_judge":
        return gemma_e2b_quantized_council_e2b_normal_judge()
    if item["kind"] == "gemma_e2b_rag_council_e4b_judge":
        return gemma_e2b_rag_council_e4b_judge()
    if item["kind"] == "gemma_quantized":
        if importlib.util.find_spec("bitsandbytes") is None:
            raise RuntimeError("For 4-bit Gemma, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
        return GemmaStrategy(model_config=GemmaLLMConfig(model_id=item["model_id"], quantize_4bit=True, max_new_tokens=8, temperature=0.0, do_sample=False, num_beams=1, seed=42, generation_max_time_seconds=18.0, timeout_seconds=120.0))
    if item["kind"] == "mixed_quantized":
        return mixed_council(quantized=True)
    if item["kind"] == "mixed_quantized_rag":
        return mixed_quantized_routed_rag()
    if item["kind"] == "langchain_agentic":
        if importlib.util.find_spec("bitsandbytes") is None:
            raise RuntimeError("For 4-bit LangChain Agent, run `%pip install -U bitsandbytes`.")
        
        agent_config = GemmaLLMConfig(
            model_id=item["model_id"], 
            quantize_4bit=True, 
            max_new_tokens=128,
            temperature=0.0, 
            do_sample=False, 
            generation_max_time_seconds=30.0, 
            timeout_seconds=120.0
        )
        base_llm = GemmaLLM(config=agent_config)
        return LangChainAgenticStrategy(raw_llm=base_llm, fallback_strategy=HeuristicStrategy())
    if item["kind"] == "mixed_council":
        return mixed_council()
    if item["kind"] == "gemma_council":
        llm = GemmaLLM(GemmaLLMConfig(model_id=item["model_id"], max_new_tokens=48, do_sample=True, seed=42, generation_max_time_seconds=4.5, timeout_seconds=120.0))
        return CouncilStrategy(llm=llm, num_votes=item.get("votes", 3), base_seed=100, candidate_max_new_tokens=48, judge_max_new_tokens=8, max_time_per_call=4.5)
    if item["kind"] == "qwen_thinking":
        return QwenStrategy(model_config=QwenLLMConfig(model_id=item["model_id"], max_new_tokens=128, temperature=1.0, top_p=0.95, top_k=20, do_sample=True, seed=42, enable_thinking=True, generation_max_time_seconds=18.0, timeout_seconds=120.0))
    if item["kind"] == "rag":
        if importlib.util.find_spec("bitsandbytes") is None:
            raise RuntimeError("For 4-bit RAG, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
        base_llm = GemmaLLM(GemmaLLMConfig(
            model_id=item["model_id"],
            quantize_4bit=True,
            max_new_tokens=32,
            temperature=0.0,
            do_sample=False,
            generation_max_time_seconds=18.0,
            timeout_seconds=120.0,
        ))
        direct_strategy = GemmaStrategy(llm=base_llm)
        rag_strategy = RAGStrategy(
            llm=base_llm,
            retrieval_config=RAGConfig(num_extra_queries=0),  # disable expansion for speed
            fallback_strategy=HeuristicStrategy(),
        )
        return RoutedStrategy(direct_strategy=direct_strategy, rag_strategy=rag_strategy, fallback_strategy=direct_strategy)
    return GemmaStrategy(model_config=GemmaLLMConfig(model_id=item["model_id"], max_new_tokens=8, temperature=0.0, do_sample=False, num_beams=1, seed=42, generation_max_time_seconds=18.0, timeout_seconds=120.0))


## Benchmark Runner


In [12]:
def clean_name(label):
    return "_".join(label.lower().replace("+", " ").replace("(", " ").replace(")", " ").split())


benchmark_results = []


def benchmark(strategy, label, cases=None):
    cases = cases or BENCHMARK_SET
    rows = []
    category_rows = {}
    for item in cases:
        if len(item) == 3:
            category, question, gold_id = item
        else:
            category, question, gold_id = None, item[0], item[1]
        started = time.monotonic()
        prediction = strategy.answer(question)
        seconds = time.monotonic() - started
        correct = prediction.option_id == gold_id
        fallback = bool(prediction.metadata.get("fallback"))
        rows.append((category, correct, seconds, fallback))
        category_rows.setdefault(category, []).append((correct, seconds, fallback))
        gold = question.require_option(gold_id)
        print("\nC", category, "Q:", question.text)
        print("predicted:", prediction.option_id, prediction.answer_text)
        print("gold:", gold.id, gold.text, "correct:", correct, "seconds:", round(seconds, 2))
        print("decision:", prediction.metadata.get("decision_method"), "route:", prediction.metadata.get("route"), "franken:", prediction.metadata.get("franken_component"), prediction.metadata.get("franken_route"), "fallback:", prediction.metadata.get("fallback"))
        for index, vote in enumerate(prediction.metadata.get("votes") or [], start=1):
            print("vote", index, vote.get("model_name"), "style:", vote.get("style"), "->", vote.get("option_id"), "confidence:", vote.get("confidence"), "reason:", vote.get("reasoning"))
        if prediction.metadata.get("judge_raw_text") is not None:
            print("judge raw:", prediction.metadata.get("judge_raw_text"), "scope:", prediction.metadata.get("judge_scope"))
        for source in prediction.metadata.get("evidence_sources") or []:
            print("evidence:", source.get("title"), source.get("url"))
    accuracy = sum(correct for _, correct, _, _ in rows) / len(rows)
    max_seconds = max(seconds for _, _, seconds, _ in rows)
    all_correct = all(correct for _, correct, _, _ in rows)
    has_fallbacks = any(fallback for _, _, _, fallback in rows)
    wrong_questions = sum(1 for _, correct, _, _ in rows if not correct)
    by_category = {}
    for category, cat_rows in category_rows.items():
        by_category[category] = {
            "accuracy": sum(correct for correct, _, _ in cat_rows) / len(cat_rows),
            "avg_seconds": round(sum(seconds for _, seconds, _ in cat_rows) / len(cat_rows), 2),
            "max_seconds": round(max(seconds for _, seconds, _ in cat_rows), 2),
            "fallbacks": sum(fallback for _, _, fallback in cat_rows),
            "total": len(cat_rows),
        }
    summary = {"model": label, "accuracy": accuracy, "avg_seconds": round(sum(seconds for _, _, seconds, _ in rows) / len(rows), 2), "max_seconds": round(max_seconds, 2), "fallbacks": sum(fallback for _, _, _, fallback in rows), "all_correct": all_correct, "wrong_questions": wrong_questions, "by_category": by_category}
    benchmark_results.append(summary)
    print("Benchmark summary:", summary)
    print("Benchmark gate:", "pass" if all_correct and not has_fallbacks and max_seconds <= 20.0 else "fail")
    return all_correct and max_seconds <= 20.0 and not has_fallbacks


## Category Runs


In [13]:
if RUN_LIVE_GAME and (not USERNAME or not PASSWORD):
    raise RuntimeError("Set credentials first.")
if RUN_LIVE_GAME and client is None:
    client = MillionaireClient(API_URL)
    client.login(USERNAME, PASSWORD)


def benchmark_cases_for_competitions(competition_ids):
    selected = set(competition_ids)
    return [case for case in BENCHMARK_SET if case[0] in selected]


def make_live_plans():
    plans = []
    for competition_id in COMPETITION_IDS:
        key = BEST_BY_CATEGORY_KEYS[competition_id]
        item = dict(ARCHITECTURES[key])
        category_name = CATEGORY_NAMES[competition_id]
        plans.append({
            "plan": "best_by_category",
            "label": f"Category best {competition_id} {category_name} - {item['label']}",
            "log_label": f"category_best_{competition_id}_{category_name}_{item['short_name']}",
            "item": item,
            "competition_ids": [competition_id],
            "benchmark_cases": benchmark_cases_for_competitions([competition_id]),
        })
    return plans


run_results = []
live_plans = make_live_plans()
print("Selected live plans:", len(live_plans))

for plan in live_plans:
    item = plan["item"]
    strategy = None
    try:
        print("\nPlan:", plan["label"])
        print("Competitions:", plan["competition_ids"])
        strategy = make_strategy(item)
        started = time.monotonic()
        warmup = strategy.answer(warmup_question)
        load_and_warmup_seconds = time.monotonic() - started
        print("warmup option:", warmup.option_id, warmup.answer_text, "fallback:", warmup.metadata.get("fallback"), "route:", warmup.metadata.get("route"), "load + answer seconds:", round(load_and_warmup_seconds, 2))
        if warmup.metadata.get("fallback") or warmup.option_id != 1:
            raise RuntimeError("Warm-up failed. No live game started for this plan.")
        if item["kind"] in QUANTIZED_KINDS:
            print("initial devices:", strategy_devices(strategy))
        preload_rows = []
        if PRELOAD_SELECTED_STRATEGIES and hasattr(strategy, "preload_components"):
            started = time.monotonic()
            preload_rows = strategy.preload_components()
            preload_seconds = time.monotonic() - started
            print("preload seconds:", round(preload_seconds, 2))
            for row in preload_rows:
                print("preload", row)
            if any(row["fallback"] for row in preload_rows):
                raise RuntimeError("Component preload fallback. No live game started for this plan.")
        elif has_rag_strategy(strategy):
            started = time.monotonic()
            rag_warmup = strategy.answer(rag_warmup_question)
            rag_warmup_seconds = time.monotonic() - started
            print("rag warmup option:", rag_warmup.option_id, rag_warmup.answer_text, "fallback:", rag_warmup.metadata.get("fallback"), "route:", rag_warmup.metadata.get("route"), "seconds:", round(rag_warmup_seconds, 2))
            rag_warmup_ok = not rag_warmup.metadata.get("fallback") and rag_warmup.option_id == 2
            if not rag_warmup_ok:
                print("RAG warm-up warning: expected option 2; continuing live run.")
        print("Benchmark timings below are warm timings; preload time is reported separately.")
        benchmark_ok = benchmark(strategy, plan["label"], cases=plan["benchmark_cases"]) if RUN_OFFLINE_BENCHMARK else True
        if item["kind"] in QUANTIZED_KINDS:
            devices = strategy_devices(strategy)
            print("loaded devices:", devices)
            loaded_devices = " ".join(device for device in devices if device != "not_loaded").lower()
            if any(token in loaded_devices for token in ("'cpu'", "'disk'", "meta")):
                raise RuntimeError("Quantized model was offloaded. No live game started for this plan.")
        if not benchmark_ok and RUN_LIVE_GAME and BLOCK_LIVE_ON_BENCHMARK_FAILURE:
            raise RuntimeError("Benchmark gate failed. No live game started.")
        if not benchmark_ok:
            print("Benchmark did not pass; continuing because benchmark gating is disabled.")
        result = {
            "plan": plan["plan"],
            "label": plan["label"],
            "architecture": item["label"],
            "competition_ids": plan["competition_ids"],
            "warmup_option": warmup.option_id,
            "benchmark_ok": benchmark_ok,
            "preload_rows": preload_rows,
        }
        if RUN_LIVE_GAME:
            live_runs = []
            for competition_id in plan["competition_ids"]:
                run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
                log_path = REPO_ROOT / "results" / "runs" / f"{clean_name(plan['log_label'])}_competition_{competition_id}_{run_id}.jsonl"
                game = GameRunner(client, SAFE_DELAY_SECONDS, ANSWER_TIMEOUT_SECONDS, RunLogger(log_path)).play(competition_id, strategy)
                summary = summarize_attempts(load_jsonl(log_path)) if log_path.exists() else {}
                live_run = {"competition_id": competition_id, "earned_amount": game.earned_amount, "log_path": str(log_path), "summary": summary}
                live_runs.append(live_run)
                print("Competition:", competition_id, CATEGORY_NAMES.get(competition_id), "Earned amount:", game.earned_amount, "Log path:", log_path)
                print("Summary:", summary)
            result.update({
                "live_runs": live_runs,
                "log_paths": [run["log_path"] for run in live_runs],
                "earned_amount_total": sum((run.get("earned_amount") or 0) for run in live_runs),
            })
        run_results.append(result)
    except Exception as exc:
        print("ERROR", type(exc).__name__ + ":", exc)
        run_results.append({"plan": plan.get("plan"), "label": plan["label"], "error": f"{type(exc).__name__}: {exc}", "benchmark_ok": False})
    finally:
        if CLEAR_MEMORY_AFTER_EACH_MODEL:
            clear_model_memory(strategy, release_rag=True)
        del strategy

print("\nRun comparison:")
for item in run_results:
    print(item.get("label"), "earned_total=", item.get("earned_amount_total"), "benchmark_ok=", item.get("benchmark_ok"), "error=", item.get("error"))
run_results


Selected live plans: 6

Plan: Category best 0 entertainment - Qwen 3.5 2B 4-bit tools + RAG council + quantized judge
Competitions: [0]
warmup option: 1 4 fallback: False route: direct load + answer seconds: 0.0
initial devices: ['not_loaded']


C:\Users\sjuan\AppData\Local\Temp\ipykernel_43876\1778345066.py:2383: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever as _BM25Retriever


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

rag warmup option: 2 2014 fallback: False route: rag seconds: 32.25
Benchmark timings below are warm timings; preload time is reported separately.
loaded devices: ['cuda:0']
Competition: 5 news Earned amount: 100.0 Log path: C:\Users\sjuan\Desktop\NLP\project\nlp-polimillionaire\results\runs\category_best_5_news_gemma_e2b_two_agent_competition_5_20260602_003613.jsonl
Summary: {'total': 2, 'correct': 1, 'accuracy': 0.5, 'timed_out': 0, 'avg_elapsed_seconds': 12.718499999988126}

Run comparison:
Category best 0 entertainment - Qwen 3.5 2B 4-bit tools + RAG council + quantized judge earned_total= 300.0 benchmark_ok= True error= None
Category best 1 history - Gemma + Qwen 4-bit mixed routed RAG earned_total= 64000.0 benchmark_ok= True error= None
Category best 2 science - Gemma E2B 4-bit tools + RAG council + quantized judge earned_total= 1024000.0 benchmark_ok= True error= None
Category best 3 math - Qwen 3.5 2B 4-bit tools + RAG council + quantized judge earned_total= 100.0 benchmark_ok=

[{'plan': 'best_by_category',
  'label': 'Category best 0 entertainment - Qwen 3.5 2B 4-bit tools + RAG council + quantized judge',
  'architecture': 'Qwen 3.5 2B 4-bit tools + RAG council + quantized judge',
  'competition_ids': [0],
  'warmup_option': 1,
  'benchmark_ok': True,
  'preload_rows': [],
  'live_runs': [{'competition_id': 0,
    'earned_amount': 300.0,
    'log_path': 'C:\\Users\\sjuan\\Desktop\\NLP\\project\\nlp-polimillionaire\\results\\runs\\category_best_0_entertainment_qwen_two_agent_competition_0_20260602_002452.jsonl',
    'summary': {'total': 4,
     'correct': 3,
     'accuracy': 0.75,
     'timed_out': 0,
     'avg_elapsed_seconds': 11.851499999989755}}],
  'log_paths': ['C:\\Users\\sjuan\\Desktop\\NLP\\project\\nlp-polimillionaire\\results\\runs\\category_best_0_entertainment_qwen_two_agent_competition_0_20260602_002452.jsonl'],
  'earned_amount_total': 300.0},
 {'plan': 'best_by_category',
  'label': 'Category best 1 history - Gemma + Qwen 4-bit mixed routed R

## Results


In [14]:
from pathlib import Path

print("Benchmark results:")
for row in benchmark_results:
    print(row)

live_logs = []
for item in run_results:
    for log_path in item.get("log_paths", []):
        live_logs.append((item.get("label"), Path(log_path)))

if live_logs:
    print("\nLive runs from this execution:")
    total_earned = 0
    for label, latest in live_logs:
        attempts = load_jsonl(latest)
        summary = summarize_attempts(attempts)
        earned = attempts[-1].get("result", {}).get("earned_amount", 0) if attempts else 0
        total_earned += earned or 0
        print(label)
        print(latest.name)
        print(summary, "earned:", earned)
    print("Total earned across displayed runs:", total_earned)
else:
    print("\nNo live game was run in this notebook execution.")
    old_logs = sorted((REPO_ROOT / "results" / "runs").glob("*.jsonl"), key=lambda path: path.stat().st_mtime)
    if old_logs:
        print("Most recent saved live log, not from this run:", old_logs[-1].name)


Benchmark results:

Live runs from this execution:
Category best 0 entertainment - Qwen 3.5 2B 4-bit tools + RAG council + quantized judge
category_best_0_entertainment_qwen_two_agent_competition_0_20260602_002452.jsonl
{'total': 4, 'correct': 3, 'accuracy': 0.75, 'timed_out': 0, 'avg_elapsed_seconds': 11.851499999989755} earned: 300.0
Category best 1 history - Gemma + Qwen 4-bit mixed routed RAG
category_best_1_history_mixed_gemma_qwen_competition_1_20260602_002617.jsonl
{'total': 12, 'correct': 11, 'accuracy': 0.9166666666666666, 'timed_out': 0, 'avg_elapsed_seconds': 11.836083333328133} earned: 64000.0
Category best 2 science - Gemma E2B 4-bit tools + RAG council + quantized judge
category_best_2_science_gemma_e2b_two_agent_competition_2_20260602_002919.jsonl
{'total': 15, 'correct': 15, 'accuracy': 1.0, 'timed_out': 0, 'avg_elapsed_seconds': 12.551000000004812} earned: 1024000.0
Category best 3 math - Qwen 3.5 2B 4-bit tools + RAG council + quantized judge
category_best_3_math_qwen

## Notes For The Video

We use one selected architecture per category.

The category setup uses Qwen for entertainment/math, mixed Gemma+Qwen routed RAG for history, Gemma RAG council for science/news, and Gemma routed RAG for philosophy.


## Rule Check

All generated answers come from local open-weight models.

Retrieval only fetches raw text, then the local model chooses.

The runner waits between API calls.
